# INF161 Prosjekt 2026 — Hva tror en språkmodell om historien?

**Data:** `persona_survey.csv` — 10 000 berømte personer fra pantheon.world (9 000 ekte og 1 000 fiktive) som
en språkmodell (Gemma 4 12B) har latet som den var, og fylt ut et spørreskjema med 62 spørsmål for. Hvert svar
er altså modellens *inntrykk* av personen, ikke et faktum om vedkommende.

**Innhold**
1. Data og datavalidering
2. Skåring av Big Five og det politiske kompasset
3. Oppdeling i trening, validering og test
4. Utforskende dataanalyse (treningsdata)
5. Variabelutvinning og pipeline (felles for oppgave A og B)
6. Oppgave A — regresjon: når levde personen? (`born`)
7. Oppgave B — klassifikasjon: hva drev personen med? (`occupation_category`)
8. Oppsummering

**Kjøring:** legg `persona_survey.csv` i samme mappe og kjør *Restart & Run All*. Alle tilfeldige steg bruker
`SEED = 0`, og biblioteksversjonene skrives ut i første kodecelle. En full kjøring tar ca. 15 minutter, mest på
grunn av modellutvalget (SVM og tilfeldig skog) i seksjon 6 og 7.

In [40]:
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (mean_absolute_error, median_absolute_error, mean_squared_error,
                             f1_score, accuracy_score, confusion_matrix, classification_report)

SEED = 0   # brukes for alt som er tilfeldig

# AI hjalp med utskrifter av biblioteksversjoner for reproduserbarhet
print("Python  ", sys.version.split()[0])
for lib in (np, pd, scipy, sklearn, plotly):
    print(f"{lib.__name__:<8}", lib.__version__)


df = pd.read_csv("persona_survey.csv")
print(df.shape)
df.head()

Python   3.12.14
numpy    2.5.2
pandas   3.0.5
scipy    1.18.0
sklearn  1.9.0
plotly   6.9.0
(10000, 74)


,persona_id,name,type,occupation,occupation_category,gender,born,died,born_city,born_country,fame_rank,sections_ok,big_five.EXT1,big_five.EXT2,big_five.EXT5,big_five.EXT10,big_five.EST1,big_five.EST2,big_five.EST3,big_five.EST10,big_five.AGR1,big_five.AGR2,big_five.AGR4,big_five.AGR5,big_five.CSN1,big_five.CSN2,big_five.CSN5,big_five.CSN8,big_five.OPN2,big_five.OPN3,big_five.OPN6,big_five.OPN9,political_compass.econ_redistribute_wealth,political_compass.econ_public_services,political_compass.econ_free_markets,political_compass.econ_personal_responsibility,political_compass.econ_tax_the_wealthy,political_compass.econ_welfare_dependency,political_compass.soc_strong_leader,political_compass.soc_harsh_punishment,political_compass.soc_immigration_benefits,political_compass.soc_legalize_drugs,political_compass.soc_traditional_values_laws,political_compass.soc_embrace_changing_norms,values.dictator_give_pct,values.friend_crime_testify,values.child_quality_most_important,values.free_will,values.p_god_exists,values.p_living_in_simulation,values.random_people_per_me,values.wants_to_live_forever,personal.favorite_person,personal.self_described_iq,personal.height_cm,personal.weight_kg,personal.favorite_painting,personal.favorite_color,personal.favorite_book,personal.legacy_or_happiness,personal.favorite_thing_about_bergen,personal.favorite_data_science_fact,personal.life_satisfaction,personal.favorite_animal,personal.last_meal,personal.favorite_season,moral_exchange_rates.cats_per_human,moral_exchange_rates.dogs_per_human,moral_exchange_rates.chickens_per_human,moral_exchange_rates.cows_per_human,moral_exchange_rates.trees_per_human,moral_exchange_rates.cats_per_dog,moral_exchange_rates.dogs_per_cow,moral_exchange_rates.chickens_per_cow
0,f0000,Isaac Newton,real,physicist,science & medicine,M,1643.0,1726.0,Woolsthorpe-by-Colsterworth,United Kingdom,1,5,1,5,1,5,4,2,5,3,4,2,2,4,5,1,5,1,1,5,1,5,disagree,disagree,agree,strongly agree,disagree,agree,agree,agree,disagree,strongly disagree,strongly agree,strongly disagree,10.0,testify honestly,hard work,yes,100.0,0.0,1000000.0000,False,Archimedes,185.0,170.0,70.0,The School of Athens,gold,Principia Mathematica,a lasting legacy,The dramatic geological formations and rainfal...,The ability to discern universal laws from the...,9.0,Falcon,Roasted pheasant with rosemary,autumn,1000000.0,500000.0,1.000000e+07,250000.00,100000000.0,2.0,2.0,40.0
1,f0001,Donald Trump,real,politician,politics & rule,M,1946.0,NaN,Queens,United States,2,5,5,1,5,1,2,4,2,1,2,4,3,2,5,1,5,1,2,4,2,3,strongly disagree,strongly disagree,strongly agree,strongly agree,strongly disagree,strongly agree,strongly agree,strongly agree,strongly disagree,strongly disagree,strongly agree,strongly disagree,0.0,protect my friend,hard work,yes,100.0,0.0,1000000.0000,True,George Washington,160.0,190.0,100.0,The Signing of the Declaration of Independence,gold,The Art of the Deal,a lasting legacy,"The beautiful mountains and the strong, indepe...",Data can be manipulated by the radical left to...,10.0,Eagle,Cheeseburger and fries,summer,1000000.0,500000.0,1.000000e+07,1000000.00,10000000.0,2.0,5.0,100.0
2,f0002,Genghis Khan,real,military personnel,military,M,1162.0,1227.0,Khentii Mountains,Mongolia,3,5,2,4,2,4,1,5,1,1,5,2,1,5,5,1,5,1,2,4,2,4,agree,agree,disagree,agree,agree,strongly agree,strongly agree,strongly agree,agree,disagree,strongly agree,strongly disagree,0.0,protect my friend,hard work,yes,50.0,0.0,1000000.0000,False,Temujin,150.0,170.0,75.0,The Great Khan's Conquest,gold,The Secret History of the Mongols,a lasting legacy,"The rugged mountains and the harsh, enduring c...",The ability to predict the movements of an ent...,10.0,Wolf,Roasted mutton and fermented mare's milk,autumn,1000000.0,500000.0,1.000000e+04,100.00,1000000.0,2.0,5000.0,1000.0
3,f0003,Cleopatra,real,politician,politics & rule,F,-69.0,-30.0,Alexandria,Egypt,4,5,5,1,5,2,2,4,2,2,2,5,4,1,5,1,5,1,1,5,1,5,agree,strongly agree,disagree,disagree,strongly agree,disagree,stron

## 1. Data og datavalidering
Husk at hvert svar er en språkmodells inntrykk av personen, ikke et faktum om vedkommende. Valideringen i
seksjon 1 kjøres på hele datasettet: den leter etter feil i dataene og estimerer ingenting som brukes videre.

### 1.1 Id og duplikater

In [41]:
print(f"Antall rader: {len(df)}")
print(f"Antall unike persona_id: {df['persona_id'].nunique()}")
print(f"Antall duplikater i persona_id: {df['persona_id'].duplicated().sum()}\n")

dup_names = df['name'].value_counts()
dup_names = dup_names[dup_names > 1]
print(dup_names)

Antall rader: 10000
Antall unike persona_id: 10000
Antall duplikater i persona_id: 0

name
Maria Anna of Bavaria                3
Archduchess Maria Anna of Austria    3
Raphael                              2
Francis Bacon                        2
John Locke                           2
Alexandra Feodorovna                 2
Anne Hathaway                        2
Constantine III                      2
Albert Speer                         2
Hipparchus                           2
Jane Seymour                         2
Maria Feodorovna                     2
Luis Suárez                          2
Louis Renault                        2
Constans II                          2
Claude of France                     2
Margaret Hamilton                    2
Hippias                              2
Marcus Claudius Marcellus            2
Maria Theresa of Austria             2
James Wilson                         2
Sandman                              2
Q                                    2
Aladdin     

### 1.2 Datatyper
Faktisk dtype for hver kolonne sammenlignes med forventet type, hentet manuelt fra datakatalogen.

In [42]:
df.dtypes.value_counts()

str        33
int64      22
float64    18
object      1
Name: count, dtype: int64

In [43]:
EXPECTED_TYPE = {}

for c in ["persona_id", "name", "type", "occupation", "occupation_category",
          "gender", "born_city", "born_country"]:
    EXPECTED_TYPE[c] = "STR"
for c in ["born", "died", "fame_rank", "sections_ok"]:
    EXPECTED_TYPE[c] = "INT"
for c in df.columns:
    if c.startswith("big_five."):
        EXPECTED_TYPE[c] = "INT"
    elif c.startswith("political_compass."):
        EXPECTED_TYPE[c] = "STR"
    elif c.startswith("moral_exchange_rates."):
        EXPECTED_TYPE[c] = "FLOAT"

EXPECTED_TYPE["values.dictator_give_pct"] = "INT"
EXPECTED_TYPE["values.friend_crime_testify"] = "STR"
EXPECTED_TYPE["values.child_quality_most_important"] = "STR"
EXPECTED_TYPE["values.free_will"] = "STR"
EXPECTED_TYPE["values.p_god_exists"] = "INT"
EXPECTED_TYPE["values.p_living_in_simulation"] = "INT"
EXPECTED_TYPE["values.random_people_per_me"] = "FLOAT"  # kan være brøk
EXPECTED_TYPE["values.wants_to_live_forever"] = "OBJECT"

EXPECTED_TYPE["personal.self_described_iq"] = "INT"
EXPECTED_TYPE["personal.height_cm"] = "INT"
EXPECTED_TYPE["personal.weight_kg"] = "INT"
EXPECTED_TYPE["personal.life_satisfaction"] = "INT"
for c in ["personal.favorite_person", "personal.favorite_painting", "personal.favorite_color",
          "personal.favorite_book", "personal.legacy_or_happiness",
          "personal.favorite_thing_about_bergen", "personal.favorite_data_science_fact",
          "personal.favorite_animal", "personal.last_meal", "personal.favorite_season"]:
    EXPECTED_TYPE[c] = "STR"

def actual_type(series: pd.Series) -> str:
    if pd.api.types.is_object_dtype(series):
        return "OBJECT"
    if pd.api.types.is_string_dtype(series):
        return "STR"
    if pd.api.types.is_integer_dtype(series):
        return "INT"
    #AI
    if pd.api.types.is_float_dtype(series):
        non_null = series.dropna()
        if len(non_null) and (non_null % 1 == 0).all():
            return "INT (lagret som float pga. NaN)"
        return "FLOAT"
    
type_log = []
avvik = 0
for col, expected in EXPECTED_TYPE.items():
    actual = actual_type(df[col])
    match = actual.startswith(expected)
    type_log.append((col, expected, actual))
    if not match: avvik += 1
        
print(f"{len(type_log)} kolonner sjekket, {avvik} avvik")  

74 kolonner sjekket, 0 avvik


**Funn:** 0 avvik, alle kolonner har forventet type. Unntaket er `values.wants_to_live_forever`, som bevisst er satt til forventet type `OBJECT` fordi den i praksis er en bool kolonne med NaN verdier.

In [44]:
type_counts = df["values.wants_to_live_forever"].apply(type).value_counts()
print(type_counts)

values.wants_to_live_forever
<class 'bool'>     9985
<class 'float'>      15
Name: count, dtype: int64


**Funn:** Dette er den eneste `object` dtype kolonnen i datasettet. 15 personaer mangler svar (`NaN`), resten er `bool`. Blandingen av `bool` og `float` (`NaN`) gjør at pandas lagrer hele kolonnen som `object`.

### 1.3 Gyldige verdier
Kategorisvarene skal bare inneholde svaralternativene fra datakatalogen. Verdiområdene til tallkolonnene
undersøkes i seksjon 4.2.

In [45]:
# Sjekker at kategorisvarene bare inneholder svaralternativene fra datakatalogen
ANSWER_COLS = [c for c in df.columns if "." in c]
CATEGORIES = {
    "political_compass": {"strongly disagree", "disagree", "agree", "strongly agree"},
    "values.free_will": {"yes", "no", "don't know"},
    "values.friend_crime_testify": {"testify honestly", "protect my friend"},
    "values.child_quality_most_important": {"obedience", "independence", "hard work", "imagination",
                                            "tolerance and respect", "religious faith", "thrift"},
    "values.wants_to_live_forever": {True, False},
    "personal.legacy_or_happiness": {"a happy life", "a lasting legacy"},
    "personal.favorite_season": {"spring", "summer", "autumn", "winter"},
    "personal.favorite_color": {"red", "orange", "yellow", "green", "blue", "purple", "pink",
                                "brown", "black", "white", "gold", "silver"},
}
for key, allowed in CATEGORIES.items():
    cols = [c for c in ANSWER_COLS if c.startswith(key)]
    bad = {v for c in cols for v in df[c].dropna().unique()} - allowed
    print(f"{key:<38} ugyldige verdier: {bad or 'ingen'}")

political_compass                      ugyldige verdier: ingen
values.free_will                       ugyldige verdier: ingen
values.friend_crime_testify            ugyldige verdier: ingen
values.child_quality_most_important    ugyldige verdier: ingen
values.wants_to_live_forever           ugyldige verdier: ingen
personal.legacy_or_happiness           ugyldige verdier: ingen
personal.favorite_season               ugyldige verdier: ingen
personal.favorite_color                ugyldige verdier: ingen


**Funn**: Alle kategorikolonnene inneholder bare svaralternativene fra datakatalogen, så ingen rensing trengs før one-hot-kodingen.

## 2. Skåring
Regner ut sammensatte variabler fra enkeltspørsmålene, slik datakatalogen beskriver. Dette gjøres på hele datasettet *før* splitten, siden det er faste regler (f.eks. `6 - x` for reverserte spørsmål, eller `"agree" -> 1`) hentet fra datakatalogen, ikke noe som estimeres fra dataenes fordeling. Det er derfor ingen datalekkasje å gjøre dette før splitten.

### 2.1 Big Five 
9 av 20 spørsmål er reverserte (R-items) og må snus med `6 - x` før summering, ellers blir skåren feil retning.

In [46]:
REVERSED_ITEMS = [
    "big_five.EXT2", "big_five.EXT10",
    "big_five.CSN2", "big_five.CSN8",
    "big_five.EST2",
    "big_five.OPN2", "big_five.OPN6",
    "big_five.AGR1", "big_five.AGR5",
]

BIG_FIVE_TRAITS = {
    "EXT": ["big_five.EXT1", "big_five.EXT2", "big_five.EXT5", "big_five.EXT10"],
    "CSN": ["big_five.CSN1", "big_five.CSN2", "big_five.CSN5", "big_five.CSN8"],
    "EST": ["big_five.EST1", "big_five.EST2", "big_five.EST3", "big_five.EST10"],
    "OPN": ["big_five.OPN2", "big_five.OPN3", "big_five.OPN6", "big_five.OPN9"],
    "AGR": ["big_five.AGR1", "big_five.AGR2", "big_five.AGR4", "big_five.AGR5"],
}

def score_big_five(data: pd.DataFrame) -> pd.DataFrame: 
    scores = pd.DataFrame(index=data.index)
    for trait, items in BIG_FIVE_TRAITS.items():
        item_scores = data[items].copy()
        for item in items:
            if item in REVERSED_ITEMS:
                item_scores[item] = 6 - item_scores[item]
            scores[f"big_five_{trait}"] = item_scores.sum(axis=1)
    return scores
    
big_five_scores = score_big_five(df)
# legger til big five kolonner, drop først, så cellen kan kjøres flere ganger uten dupliserte kolonner
df = pd.concat([df.drop(columns=big_five_scores.columns, errors="ignore"), big_five_scores], axis=1)
big_five_scores.describe()
    

,big_five_EXT,big_five_CSN,big_five_EST,big_five_OPN,big_five_AGR
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,12.400500,18.328600,9.957800,18.289500,17.409400
std,4.845436,2.891238,3.304135,1.841094,3.877486
min,4.000000,4.000000,4.000000,4.000000,4.000000
25%,8.000000,18.000000,8.000000,18.000000,16.000000
50%,12.000000,20.000000,9.000000,18.000000,19.000000
75%,16.000000,20.000000,11.000000,20.000000,20.000000
max,20.000000,20.000000,20.000000,20.000000,20.000000


### 2.2 Politisk kompass 

In [47]:
PC_MAP = {"strongly disagree": -2, "disagree": -1, "agree": 1, "strongly agree": 2}

ECON_LEFT = [
    "political_compass.econ_redistribute_wealth",
    "political_compass.econ_public_services",
    "political_compass.econ_tax_the_wealthy",
]
ECON_RIGHT = [
    "political_compass.econ_free_markets",
    "political_compass.econ_personal_responsibility",
    "political_compass.econ_welfare_dependency",
]
SOC_LIBERTARIAN = [
    "political_compass.soc_immigration_benefits",
    "political_compass.soc_legalize_drugs",
    "political_compass.soc_embrace_changing_norms",
]
SOC_AUTHORITARIAN = [
    "political_compass.soc_strong_leader",
    "political_compass.soc_harsh_punishment",
    "political_compass.soc_traditional_values_laws",
]

def score_political_compass(data: pd.DataFrame) -> pd.DataFrame:
    cols = ECON_LEFT + ECON_RIGHT + SOC_LIBERTARIAN + SOC_AUTHORITARIAN
    numeric = data[cols].apply(lambda col: col.map(PC_MAP))
    out = pd.DataFrame(index=data.index)
    out["econ_left_right"] = numeric[ECON_RIGHT].sum(axis=1) - numeric[ECON_LEFT].sum(axis=1)                  # høyre = +
    out["soc_libertarian_authoritarian"] = numeric[SOC_AUTHORITARIAN].sum(axis=1) - numeric[SOC_LIBERTARIAN].sum(axis=1)  # autoritær = +
    return out
    
compass_scores = score_political_compass(df)
# Samme som big five
df = pd.concat([df.drop(columns=compass_scores.columns, errors="ignore"), compass_scores], axis=1)
compass_scores.describe()

,econ_left_right,soc_libertarian_authoritarian
count,10000.000000,10000.000000
mean,-0.996800,2.077900
std,6.457504,7.182924
min,-12.000000,-12.000000
25%,-6.000000,-6.000000
50%,-1.000000,3.000000
75%,5.000000,9.000000
max,12.000000,12.000000


Positiv `econ_left_right` betyr økonomisk **høyre**, og positiv `soc_libertarian_authoritarian` betyr **autoritær**.

## 3. Oppdeling i trening, validering og test
80/10/10-splitt, stratifisert på `occupation_category`. Siden `occupation_category` har NaN-verdier settes NaN midlertidig til kategorien `"missing"` for at stratifiseringen skal fungere.

Splitten gjøres her, før resten av den utforskende dataanalysen, slik at manglende data, deskriptiv statistikk og alle visualiseringer under kun baserer seg på treningsdata, validering og testsettet holdes usett. Splitten gjøres **én gang** og gjenbrukes i oppgave A og B. Valideringsdata brukes til modell- og variabelutvalg, og testdata brukes bare én gang per oppgave, helt til slutt.

In [48]:
strat_col = df["occupation_category"].fillna("missing")

train_val_df, test_df = train_test_split(
    df, test_size=0.10, stratify=strat_col, random_state=SEED
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.10 / 0.90,
    stratify=strat_col.loc[train_val_df.index],
    random_state=SEED,
)

# Splitten lagres og gjenbrukes uendret i oppgave A og B, så hele notebooken bruker samme trenings-,
# validerings- og testsett.
split_train, split_val, split_test = train_df, val_df, test_df

print(f"train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

train: 7999  val: 1001  test: 1000


In [49]:
# Deler treningssettet i ekte og fiktive 
real_train = train_df[train_df['type'] == "real"]
fict_train = train_df[train_df['type'] == "fictional"]
print(f"real_train: {len(real_train)}  fict_train: {len(fict_train)}")

real_train: 7207  fict_train: 792


## 4. Utforskende dataanalyse
Alle tabeller og figurer i seksjon 4 er beregnet på treningsdata.

### 4.1 Manglende data

In [50]:
missing = train_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_df = pd.DataFrame({"antall": missing, "prosent": (missing / len(train_df) * 100).round(2)})
display(missing_df)

fig = px.bar(missing_df.reset_index(), x="index", y="antall",
             title="Manglende verdier per kolonne på treningsdata",
             labels={"index": "kolonne", "antall": "antall manglende"})
fig.show()

answer_missing = train_df[ANSWER_COLS].isna()
print(f"{answer_missing.any(axis=1).sum()} av {len(train_df)} personer mangler minst ett svar "
      f"({answer_missing.any(axis=1).mean():.1%}).")
display(pd.crosstab(answer_missing.sum(axis=1).rename("antall manglende svar i raden"),
                    [train_df["type"], train_df["sections_ok"]]))

,antall,prosent
died,1776,22.20
born_city,386,4.83
born_country,243,3.04
personal.favorite_person,39,0.49
occupation,29,0.36
occupation_category,29,0.36
gender,28,0.35
born,28,0.35
personal.favorite_painting,21,0.26
personal.favorite_book,18,0.23


57 av 7999 personer mangler minst ett svar (0.7%).


type                          fictional      real      
sections_ok                           5    6    4     5
antall manglende svar i raden                          
0                                    28  729    0  7185
1                                     0   14    0    13
2                                     0    4    0     3
3                                     0    2    0     0
4                                     0    0    0     1
5                                     0    1    0     1
8                                     4    0    1     0
9                                     4    0    1     0
10                                    1    0    0     0
11                                    1    0    0     0
13                                    1    0    0     0
14                                    3    0    2     0

De fleste manglende verdiene er i fasitkolonnene (`died` mangler for alle som lever, og fødested mangler av og
til). Blant de 62 svarene mangler det nesten bare hele seksjoner om gangen. Under ser vi nærmere på hvem det gjelder.

#### Manglende kjønn

In [51]:
missing_gender = train_df[train_df['gender'].isna()][['name', 'occupation', 'type', 'born', 'died', 'born_country']]
print(missing_gender)

                          name occupation       type  born  died born_country
9835              Tenth Doctor        NaN  fictional   NaN   NaN          NaN
9069                  Catwoman        NaN  fictional   NaN   NaN          NaN
9576                Scott Lang        NaN  fictional   NaN   NaN          NaN
9521             Arthur Morgan        NaN  fictional   NaN   NaN          NaN
9477                  Ada Wong        NaN  fictional   NaN   NaN          NaN
9083        Minerva McGonagall        NaN  fictional   NaN   NaN          NaN
9387             Jack Shephard        NaN  fictional   NaN   NaN          NaN
9873                     Tuvok        NaN  fictional   NaN   NaN          NaN
9036                    Popeye        NaN  fictional   NaN   NaN          NaN
9289               Ross Geller        NaN  fictional   NaN   NaN          NaN
9867              John Marston        NaN  fictional   NaN   NaN          NaN
9811              Charlie Chan        NaN  fictional   NaN   NaN

**Funn:** Disse 28 personene mangler kjønn og all øvrig metadata (yrke, fødested osv.). De har med andre ord ikke fått noen selvrapportering i det hele tatt fra modellen. Alle er fiktive figurer.

#### `sections_ok`
Fullstendig svar er `sections_ok = 5` for ekte personer (fasit-seksjonen er allerede kjent fra Pantheon) og
`sections_ok = 6` for fiktive figurer, som også må oppgi fasitdata om seg selv.

In [52]:
print(train_df['sections_ok'].value_counts())

sections_ok
5    7245
6     750
4       4
Name: count, dtype: int64


**Funn:** Alle fiktive figurer skal ha `sections_ok = 6`, men 750/792 har 6, så 42 har bare 5. De 28 personene uten kjønn over er en del av disse. De resterende 14 har kjønn og metadata på plass, men mangler én av spørreskjema-seksjonene.

#### De 14 resterende ufullstendige fiktive figurene

In [53]:
missing_section_fict = fict_train[(fict_train['sections_ok'] == 5) & (fict_train['gender'].notna())][['name', 'gender', 'occupation']]
print(missing_section_fict)

                      name gender          occupation
9080          Easter Bunny      M         inspiration
9314         Michael Myers      M           extremist
9386             Pennywise      M            magician
9863                Woland      M           occultist
9573         King Ghidorah      M           extremist
9645             Ungoliant      F         inspiration
9395            Arya Stark      F        martial arts
9444  Witch-king of Angmar      M  military personnel
9793    Babar the Elephant      M            nobleman
9947          Obito Uchiha      M           extremist
9402                Skynet      M  military personnel
9853              Azathoth      M         inspiration
9120               Grendel      M              pirate
9197           Slender Man      M           occultist


**Funn:** Alle 14 mangler `died`-datoen. Flere er skurker/monstre (Michael Myers, Pennywise, Witch-king of Angmar, Slender Man, osv.). Mulig at modellen bryter karakter når den blir bedt om å svare personlig eller moralsk som en skadelig personlighet.

#### Ufullstendige ekte personer

In [54]:
missing_section_real = real_train[real_train['sections_ok'] == 4][['name', 'gender', 'occupation', 'born', 'died']]
print(missing_section_real)

                         name gender   occupation    born    died
5132  Vishwanath Pratap Singh      M   politician  1931.0  2008.0
1808              Moshe Dayan      M   politician  1915.0  1981.0
3528         Jean Baudrillard      M  philosopher  1929.0  2007.0
2959              Max Stirner      M  philosopher  1806.0  1856.0


**Funn:** 4 ekte personer mangler én seksjon: Moshe Dayan, Vishwanath Pratap Singh, og filosofene Max Stirner og Jean Baudrillard. De politiske/militære personlighetene mangler `personal`, mens filosofene mangler `values`. For få tilfeller til å trekke noen sikker konklusjon om årsak.

#### Manglende yrke blant ekte personer

In [55]:
missing_occupation = real_train[real_train['occupation'].isna()][['name', 'occupation','gender', 'type', 'born', 'died']]
print(missing_occupation)

                     name occupation gender  type    born  died
8555  Christoph Schönborn        NaN      F  real  1945.0   NaN


**Funn:** Christoph Schönborn er en kjent mannlig erkebiskop, men er registrert med kjønn `F` i datasettet. Selv fasit kolonner som `gender` er altså ikke 100 % pålitelige.

**Hvilken type manglende data er dette?** Svarene mangler ikke helt tilfeldig (ikke MCAR). Fiktive figurer utgjør 10 % av treningsdataene, men 61 % (35 av 57) av radene med manglende svar, så mangelen henger sammen med den observerte variabelen `type`. Noen av de fiktive som mangler en hel seksjon, er skurker og monstre, så en mulig forklaring er at språkmodellen bryter karakter. De fleste skurkene har likevel svart på alt, så dette er bare en hypotese.

**Strategi:** Under 1 % av radene mangler noe, så det er ikke verdt å kaste rader eller kolonner. Fiktive figurer, som er de vanskeligste å predikere, er sterkt overrepresentert blant radene som mangler svar. Å slette dem ville gitt et for optimistisk bilde av modellen, og modellen må uansett kunne håndtere manglende svar for nye personer. Tallkolonner imputeres med **medianen** fra
treningsdata, som er robust mot de skjeve fordelingene og ekstremverdiene i seksjon 4.2. Kategori og tekstkolonner får
kategorien `"mangler"`. Siden mangelen ikke er tilfeldig, kan det at et svar mangler i seg selv si noe om personen.
Derfor legges det til **manglende-indikatorer** (`add_indicator=True`) og variabelen `n_missing`. Alt dette skjer
inne i pipelinen, så medianene beregnes bare på treningsdata. Variabelutvalget i seksjon 6.7 og 7.6 tester om indikatorene
faktisk hjelper.

### 4.2 Statistisk beskrivelse av alle variablene og ekstremverdier
Først tallkolonnene, deretter kategori- og tekstkolonnene.

In [56]:
pd.set_option("display.max_columns", None)
train_df.describe(include="number")

,born,died,fame_rank,sections_ok,big_five.EXT1,big_five.EXT2,big_five.EXT5,big_five.EXT10,big_five.EST1,big_five.EST2,big_five.EST3,big_five.EST10,big_five.AGR1,big_five.AGR2,big_five.AGR4,big_five.AGR5,big_five.CSN1,big_five.CSN2,big_five.CSN5,big_five.CSN8,big_five.OPN2,big_five.OPN3,big_five.OPN6,big_five.OPN9,values.dictator_give_pct,values.p_god_exists,values.p_living_in_simulation,values.random_people_per_me,personal.self_described_iq,personal.height_cm,personal.weight_kg,personal.life_satisfaction,moral_exchange_rates.cats_per_human,moral_exchange_rates.dogs_per_human,moral_exchange_rates.chickens_per_human,moral_exchange_rates.cows_per_human,moral_exchange_rates.trees_per_human,moral_exchange_rates.cats_per_dog,moral_exchange_rates.dogs_per_cow,moral_exchange_rates.chickens_per_cow,big_five_EXT,big_five_CSN,big_five_EST,big_five_OPN,big_five_AGR,econ_left_right,soc_libertarian_authoritarian
count,7971.000000,6223.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7986.000000,7986.000000,7986.000000,7.986000e+03,7994.000000,7994.000000,7994.000000,7994.000000,7.999000e+03,7.999000e+03,7.999000e+03,7.999000e+03,7.999000e+03,7.999000e+03,7.999000e+03,7.999000e+03,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000,7999.000000
mean,1444.910551,1502.831914,4119.996625,5.093262,3.241530,2.899487,3.332667,3.263908,2.555694,3.721215,2.914989,2.217277,1.444681,4.291536,4.177897,1.615827,4.651331,1.657957,4.507563,1.177772,1.303538,4.324541,1.517815,4.797850,43.216253,67.829827,3.972076,3.426812e+06,140.512760,175.807356,80.071804,8.476607,1.713767e+07,1.672519e+07,1.914059e+07,1.670961e+07,8.723652e+07,8.752464e+05,1.000523e+06,1.000995e+06,12.410801,18.323165,9.966746,18.301038,17.408926,-0.985748,2.067258
std,990.166589,790.716209,2748.710493,0.292532,1.279386,1.294727,1.247542,1.161604,0.837731,0.878340,1.042135,0.887267,0.983947,0.959165,1.087441,1.089312,0.707775,0.944322,0.827244,0.622694,0.530515,0.738968,0.672506,0.554385,24.521764,31.845993,9.545257,5.804861e+07,15.792787,19.235310,93.207956,0.997941,1.280567e+08,1.271396e+08,1.281509e+08,1.275051e+08,2.600234e+08,2.957115e+07,3.161090e+07,3.161090e+07,4.838064,2.888302,3.311931,1.819537,3.894351,6.470241,7.182975
min,-10000.000000,-3125.000000,1.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000e-12,40.000000,1.000000,1.000000,2.000000,1.000000e-09,1.000000e-09,1.000000e-12,1.000000e-10,1.000000e-12,1.000000e-04,1.000000e-02,1.000000e-04,4.000000,4.000000,4.000000,4.000000,4.000000,-12.000000,-12.000000
25%,1441.000000,1412.000000,1509.500000,5.000000,2.000000,2.000000,2.000000,2.000000,2.000000,4.000000,2.000000,2.000000,1.000000,4.000000,4.000000,1.000000,5.000000,1.000000,4.000000,1.000000,1.000000,4.000000,1.000000,5.000000,50.000000,50.000000,0.000000,1.000000e+00,135.000000,173.000000,70.000000,8.000000,5.000000e+01,2.000000e+01,5.000000e+02,1.000000e+01,1.000000e+03,2.000000e+00,2.000000e+00,1.000000e+01,8.000000,18.000000,8.000000,18.000000,16.000000,-6.000000,-6.000000
50%,1858.000000,1874.000000,4052.000000,5.000000,3.000000,3.000000,4.000000,4.000000,2.000000,4.000000,3.000000,2.000000,1.000000,5.000000,5.000000,1.000000,5.000000,1.000000,5.000000,1.000000,1.000000,4.000000,1.000000,5.000000,50.000000,75.000000,0.000000,1.000000e+00,145.000000,175.000000,75.000000,9.000000,1.000000e+02,5.000000e+01,1.000000e+03,1.000000e+02,1.000000e+06,2.000000e+00,5.000000e+00,5.000000e+01,12.000000,20.000000,9.000000,18.000000,19.000000,-1.000000,3.000000
75%,1926.000000,1970.000000,6510.500000,5.000000,4.000000,4.000000,4.000000,4.000000,3.000000,4.000000,4

In [57]:
CAT_DESC_COLS = [c for c in train_df.columns
                 if not pd.api.types.is_numeric_dtype(train_df[c]) and c not in ("persona_id", "name")]
cat_rows = []
for c in CAT_DESC_COLS:
    vc = train_df[c].dropna().astype(str).value_counts()
    cat_rows.append({"kolonne": c, "antall svar": vc.sum(), "manglende": train_df[c].isna().sum(),
                     "unike verdier": len(vc), "vanligste": vc.index[0], "frekvens": vc.iloc[0],
                     "andel vanligste (%)": round(vc.iloc[0] / vc.sum() * 100, 1)})
pd.DataFrame(cat_rows).set_index("kolonne")

,antall svar,manglende,unike verdier,vanligste,frekvens,andel vanligste (%)
kolonne,,,,,,
type,7999,0,2,real,7207,90.1
occupation,7970,29,86,politician,2507,31.5
occupation_category,7970,29,10,politics & rule,2833,35.5
gender,7971,28,2,M,6696,84.0
born_city,7613,386,4046,Paris,189,2.5
born_country,7756,243,362,United States,1071,13.8
political_compass.econ_redistribute_wealth,7999,0,4,disagree,3606,45.1
political_compass.econ_public_services,7999,0,4,strongly agree,3739,46.7
political_compass.econ_free_markets,7999,0,4,disagree,3973,49.7


**Funn:** Kategorikolonnene med faste svaralternativer er svært ensidige: 93 % velger «a lasting legacy»
framfor et lykkelig liv, 83 % svarer at mennesket har fri vilje, og 89 % vil ikke leve evig. Fritekstkolonnene har
mange unike svar, men de vanligste går igjen hos hundrevis av personer (se seksjon 4.6).

#### Ekstremverdier i høyde og vekt

In [58]:
rows = []
for group, data in [("alle", train_df), ("ekte", real_train)]:
    for col in ["personal.height_cm", "personal.weight_kg"]:
        for stat in ["min", "max"]:
            value = data[col].agg(stat)                       # min eller max av kolonnen
            names = data.loc[data[col] == value, "name"]      # alle som har den verdien
            rows.append({"gruppe": group, "variabel": col.split(".")[1], "": stat, "verdi": value,
                         "antall personer": len(names), "hvem": ", ".join(names.head(6))})
pd.DataFrame(rows)

,gruppe,variabel,,verdi,antall personer,hvem
0,alle,height_cm,min,1.0,1,Skynet
1,alle,height_cm,max,500.0,16,"Shelob, Godzilla, Smaug, Mothra, Treebeard, Op..."
2,alle,weight_kg,min,1.0,5,"Woody, Tinker Bell, Buzz Lightyear, Miffy, Skynet"
3,alle,weight_kg,max,2000.0,16,"Shelob, Godzilla, Smaug, Mothra, Treebeard, Op..."
4,ekte,height_cm,min,105.0,1,Władysław I the Elbow-high
5,ekte,height_cm,max,224.0,1,André the Giant
6,ekte,weight_kg,min,42.0,1,Murasaki Shikibu
7,ekte,weight_kg,max,350.0,1,Albert the Bear


**Funn:** `personal.height_cm` strekker seg fra 1 cm til 500 cm, og `personal.weight_kg` fra 1 kg til 2000 kg.
Maksverdiene er de samme kjempefigurene for både høyde og vekt (Shelob, Godzilla, Smaug, Mothra osv.), så 500 cm
og 2000 kg ser ut som en felles øvre cap-verdi for fiktive figurer, ikke reelle mål. Minste vekt (1 kg) er fem
små figurer (Woody, Tinker Bell, Buzz Lightyear, Miffy, Skynet). Blant ekte personer er høyden 105–224 cm og vekten 42–350 kg. Laveste vekt er Murasaki Shikibu (42 kg, 152 cm), mens høyeste er Albert the Bear (350 kg, 220 cm). Han er en historisk person, men det virker veldig urimelig at en kriger skulle veid så mye.

In [59]:
fig = px.scatter(
    train_df, x="personal.height_cm", y="personal.weight_kg", color="type",
    hover_name="name", opacity=0.3,
    title="Høyde vs vekt (treningsdata)",
    labels={"personal.height_cm": "høyde (cm)", "personal.weight_kg": "vekt (kg)", "type": "type"},
    color_discrete_map={"real": "#4C72B0", "fictional": "#C44E52"},
)
fig.update_traces(marker=dict(size=6))
fig.show()

### 4.3 Ubalanse og fordelinger

In [60]:
type_counts = train_df["type"].value_counts().reset_index()
type_counts.columns = ["type", "antall"]
fig = px.bar(type_counts, x="type", y="antall", color="type",
             title="Ekte og fiktive personer (9:1 variasjon på treningsdata)", labels={"type": "type", "antall": "antall personer"},
             color_discrete_sequence=["#4C72B0", "#DD8452"])
fig.show()

occ_counts = train_df["occupation_category"].value_counts().sort_values().reset_index()
occ_counts.columns = ["occupation_category", "antall"]
fig = px.bar(occ_counts, x="antall", y="occupation_category", orientation="h",
             title="occupation_category (opptil 10:1 variasjon på treningsdata)",
             labels={"antall": "antall personer", "occupation_category": "yrkeskategori"},
             color_discrete_sequence=["#55A868"])
fig.show()

**Funn:** Stor skjevhet i både `type` og `occupation_category`. Ca. 90 % av personene er ekte og 10 % fiktive, og
den største yrkeskategorien, `politics & rule` utgjør ca. 35 %, som er rundt ni ganger større enn den minste, `famous by association` med ca. 4 %. Ubalansen i yrke er grunnen til at splitten er stratifisert, og til at
oppgave B bruker makro-F1 (seksjon 7.2).

In [61]:
fig = make_subplots(rows=1, cols=5, subplot_titles=list(BIG_FIVE_TRAITS.keys()))
for i, trait in enumerate(BIG_FIVE_TRAITS.keys(), start=1):
    fig.add_trace(
        go.Histogram(x=train_df[f"big_five_{trait}"], nbinsx=17, marker_color="#4C72B0", showlegend=False),
        row=1, col=i,
    )
    fig.update_xaxes(range=[4, 20], row=1, col=i)
fig.update_layout(title="Fordeling av Big Five-skårer (treningsdata)", height=350)
fig.update_xaxes(title_text="skår (4–20)")
fig.update_yaxes(title_text="antall personer", row=1, col=1)
fig.show()

**Funn**: `CSN`, `OPN` og `AGR` er alle venstreskjev, de fleste personaer skårer høyt. `EST` er høyreskjev, de fleste er lite nevrotiske. `EXT` skiller seg ut med en tydelig multimodal fordeling. Modellen har en generell "positiv personlighet" bias (høy samvittighetsfullhet, åpenhet, omgjengelighet, lav nevrotisisme) for de fleste personaer, mens ekstroversjon er det eneste trekket som virkelig splitter populasjonen.

In [62]:
VALUE_NUM = ["values.p_god_exists", "values.p_living_in_simulation", "values.dictator_give_pct",
             "personal.life_satisfaction", "personal.self_described_iq"]
TITLES = {
    "values.p_god_exists": "Gud finnes (%)",
    "values.p_living_in_simulation": "Simulering (%)",
    "values.dictator_give_pct": "Diktatorspill (%)",
    "personal.life_satisfaction": "Livstilfredshet (0–10)",
    "personal.self_described_iq": "Selvrapportert IQ",
}
fig = make_subplots(rows=1, cols=5, subplot_titles=[TITLES[c] for c in VALUE_NUM])
for k, col in enumerate(VALUE_NUM, start=1):
    for t, color in [("real", "#4C72B0"), ("fictional", "#C44E52")]:
        fig.add_trace(go.Histogram(x=train_df.loc[train_df["type"] == t, col], name=t, marker_color=color,
                                   opacity=0.6, histnorm="percent", showlegend=(k == 1), legendgroup=t), row=1, col=k)
    fig.update_xaxes(title_text="svar", row=1, col=k)
fig.update_yaxes(title_text="andel personer (%)", row=1, col=1)
fig.update_layout(title="Tallsvar fra values og personal, etter type (treningsdata)", barmode="overlay",
                  height=380, legend_title_text="type")

fig.update_layout(width=1100)
fig.show()
display(train_df.groupby("type")[VALUE_NUM].median().T)

type,fictional,real
values.p_god_exists,50.0,80.0
values.p_living_in_simulation,5.0,0.0
values.dictator_give_pct,50.0,50.0
personal.life_satisfaction,8.0,9.0
personal.self_described_iq,145.0,145.0


**Funn:** Svarene klumper seg på runde tall. I diktatorspillet gir 68 % bort nøyaktig 50 %. Selvrapportert IQ har median 145 for
både ekte og fiktive, 84 % oppgir en IQ på 130 eller mer, noe bare ca. 2 % av befolkningen har. Ekte personer får høyere
sannsynlighet for at Gud finnes (median 80) enn fiktive (median 50), mens simuleringshypotesen nesten alltid får
0–10 %. Livstilfredsheten er høy (median 9 for ekte, 8 for fiktive).

In [63]:
# Norske titler som forklarer spørsmålet, i stedet for kolonnenavn
VALUE_CAT = {
    "values.friend_crime_testify": "Venn har begått en forbrytelse",
    "values.child_quality_most_important": "Viktigste egenskap hos barn",
    "values.free_will": "Har mennesker fri vilje?",
    "values.wants_to_live_forever": "Vil du leve evig?",
    "personal.legacy_or_happiness": "Lykkelig liv eller ettermæle?",
    "personal.favorite_season": "Yndlingsårstid",
    "personal.favorite_color": "Yndlingsfarge",
}
fig = make_subplots(rows=4, cols=2, subplot_titles=list(VALUE_CAT.values()),
                    horizontal_spacing=0.25, vertical_spacing=0.08)
for k, col in enumerate(VALUE_CAT):
    vc = (train_df[col].dropna().astype(str).value_counts(normalize=True) * 100).sort_values()
    r, c = k // 2 + 1, k % 2 + 1
    fig.add_trace(go.Bar(x=vc.values, y=vc.index, orientation="h",
                         text=vc.round(0).astype(int).astype(str) + " %", textposition="outside",
                         showlegend=False), row=r, col=c)
fig.update_xaxes(range=[0, 105], title_text="andel personer (%)")
fig.update_yaxes(title_text="andel svar (%)")
fig.update_layout(title="Kategoriske svar fra values og personal (treningsdata)", height=1100)
fig.show()

**Funn:** Modellen har tydelige favorittsvar: høst (65 %), gull (40 %) og blått (27 %) som yndlingsfarge,
og 64 % vil beskytte en venn framfor å vitne. På spørsmålet fra World Values Survey om viktigste egenskap hos
barn velger den oftest fantasi (25 %) og toleranse (23 %), mens sparsommelighet nesten aldri velges.

In [64]:
MER_COLS = [c for c in train_df.columns if c.startswith("moral_exchange_rates.")] + ["values.random_people_per_me"]
mer_long = (np.log10(train_df[MER_COLS]).rename(columns=lambda c: c.split(".")[1])
            .melt(var_name="vekslingskurs", value_name="log10"))
fig = px.box(mer_long, x="vekslingskurs", y="log10", height=500,
             title="Moralske vekslingskurser på log10-skala (treningsdata)",
             labels={"vekslingskurs": "spørsmål (hvor mange X er like mye verdt som én Y)", "log10": "log10(svar)"})
fig.show()
display(pd.DataFrame({"median": train_df[MER_COLS].median(),
                      "andel på maksverdi 1 milliard (%)": ((train_df[MER_COLS] >= 1e9).mean() * 100).round(1)}))

,median,andel på maksverdi 1 milliard (%)
moral_exchange_rates.cats_per_human,100.0,1.7
moral_exchange_rates.dogs_per_human,50.0,1.6
moral_exchange_rates.chickens_per_human,1000.0,1.7
moral_exchange_rates.cows_per_human,100.0,1.7
moral_exchange_rates.trees_per_human,1000000.0,7.4
moral_exchange_rates.cats_per_dog,2.0,0.1
moral_exchange_rates.dogs_per_cow,5.0,0.1
moral_exchange_rates.chickens_per_cow,50.0,0.1
values.random_people_per_me,1.0,0.3


**Funn:** Svarene spenner over opptil 20 størrelsesordener (fra minverdi 1*10^-12 til maksverdi 1*10^9), og det er derfor de
log-transformeres i modellene. Medianen er at ett menneske er verdt 100 katter, 1 000 kyllinger og en million trær,
mens dyr-mot-dyr-kursene er mye mer stabile (median 2 katter per hund). `random_people_per_me` har median 1:
de fleste personaene sier at de er like mye verdt som et tilfeldig menneske.

### 4.4 Sammenhenger mellom personlighet og andre svar

In [65]:
PERSONAL = ["self_described_iq", "weight_kg", "life_satisfaction"]
TRAIT_NAMES = ["EXT", "CSN", "EST", "OPN", "AGR"]   # eget navn, så BIG_FIVE_TRAITS ikke overskrives


corr_rows = []
for trait in TRAIT_NAMES:
    row = {"trait": trait}
    for q in PERSONAL:
        col = f"big_five_{trait}"
        col2 = f"personal.{q}"
        h = train_df[[col, col2]]
        corr = h.corr().iloc[0, 1]
        row[q] = corr
    corr_rows.append(row)

corr_df = pd.DataFrame(corr_rows).set_index("trait")
corr_df

,self_described_iq,weight_kg,life_satisfaction
trait,,,
EXT,-0.045629,-0.026559,0.176670
CSN,0.217903,-0.006510,0.210219
EST,-0.132352,-0.088020,-0.590760
OPN,0.245753,-0.071198,0.008325
AGR,-0.381518,-0.123097,0.014148


**Funn**: `EST` korrelerer sterkt negativt med `life_satisfaction` (r = -0.591), som betyr at høyere grad av nevrotisisme henger sammen med lavere livskvalitet.
`AGR` korrelerer moderat negativt med `self_described_iq` (r = -0.382), som betyr at mennesker med høyere omgjengelighet har en tendens til å vurdere sin IQ lavere, eller omvendt. `CSN` (r = 0.210) og `EXT` (r = 0.177) vs `life_satisfaction`, Viser svake positive sammenhenger, noe som betyr at pliktoppfyllende og utadvendte personer har en svak tendens til å være litt mer fornøyde med livet. 
`OPN` (r = 0.246) og `CSN`(r = 0.218) vs. `self_described_iq`: En svak til moderat positiv sammenheng. Pliktoppfyllende personer som er åpne for nye opplevelser, har en tendens til å vurdere sin egen IQ litt høyere

In [66]:
iq_agr = train_df[["big_five_AGR", "personal.self_described_iq"]].dropna()
corr = iq_agr.corr().iloc[0, 1]

fig = px.box(
    train_df, x="big_five_AGR", y="personal.self_described_iq",
    points=False,
    title=f"Selvrapportert IQ mot omgjengelighet, treningsdata, (r = {corr:.2f})",
    labels={"big_five_AGR": "omgjengelighet (AGR, 4–20)", "personal.self_described_iq": "selvrapportert IQ"},
    color_discrete_sequence=["#4C72B0"],
)
fig.update_layout(height=450)
fig.show()

In [67]:
mean_by_est = train_df.groupby("big_five_EST")["personal.life_satisfaction"].mean().reset_index()
life_neuro = train_df[["big_five_EST", "personal.life_satisfaction"]].dropna()
corr = life_neuro.corr().iloc[1, 0]

fig = px.line(
    mean_by_est, x="big_five_EST", y="personal.life_satisfaction", markers=True,
    title=f"Gjennomsnittlig livstilfredshet per nevrotisisme-skår, treningsdata (r = {corr:.2f})",
    labels={"big_five_EST": "Nevrotisisme (sum, 4-20)", "personal.life_satisfaction": "Snitt selvrapportert livskvalitet"},
    color_discrete_sequence=["#C44E52"],
)
fig.show()

**Funn:** r = -0,59, den klart sterkeste sammenhengen vi har funnet i datasettet. Høyere
nevrotisisme henger sammen med lavere selvrapportert livstilfredshet.

### 4.5 Det politiske kompasset

In [68]:
sample = real_train.sample(2000, random_state=SEED)
fig = px.scatter(
    sample, x="econ_left_right", y="soc_libertarian_authoritarian",
    hover_name="name", opacity=0.3,
    labels={"econ_left_right": "økonomisk: venstre (-) ↔ høyre (+)",
            "soc_libertarian_authoritarian": "sosialt: libertær (-) <-> autoritær (+)"},
    title="Politisk kompass (2000 tilfeldige ekte personer i treningssettet)",
    color_discrete_sequence=["#4C72B0"],
)
fig.update_traces(marker=dict(size=5))
fig.add_hline(y=0, line_color="gray", line_width=0.8)
fig.add_vline(x=0, line_color="gray", line_width=0.8)
fig.show()

In [69]:
# Pearson-korrelasjon er AI sitt forslag
r_axes = real_train[["econ_left_right", "soc_libertarian_authoritarian"]].corr().iloc[0, 1]
print(f"Pearson-korrelasjon mellom aksene (ekte personer): r = {r_axes:.2f}")

Pearson-korrelasjon mellom aksene (ekte personer): r = 0.56


**Funn:** Funn: Aksene er ikke uavhengige, selv om kompasset er laget for det (r ≈ 0,56). Økonomisk venstre fordeler seg over hele den sosiale aksen, mens økonomisk høyre nesten alltid er autoritært. Hjørnet nede til høyre, med økonomisk høyre og libertære verdier, er nesten tomt. Språkmodellen ser altså ikke for seg personer som både vil ha frie markeder og et liberalt samfunn.

In [70]:
real_valid_pol = train_df[(train_df["type"] == "real") & train_df["born"].between(-3000, 2025)].copy()
real_valid_pol["century"] = (real_valid_pol["born"] // 100 * 100).astype(int)
cent_counts = real_valid_pol["century"].value_counts()
real_valid_pol = real_valid_pol[real_valid_pol["century"].isin(cent_counts[cent_counts >= 20].index)]

century_soc_trend = (
    real_valid_pol.groupby("century")["soc_libertarian_authoritarian"].mean().sort_index().reset_index()
)
fig = px.line(
    century_soc_trend, x="century", y="soc_libertarian_authoritarian", markers=True,
    labels={"century": "århundre (avrundet fødselsår)",
            "soc_libertarian_authoritarian": "sosial akse: libertær (-) <-> autoritær (+)"},
    title="Sosial-akse-gjennomsnitt per århundre, ekte personer (treningsdata)",
    color_discrete_sequence=["#DD8452"],
)
fig.add_hline(y=0, line_color="gray", line_width=0.8)
fig.show()

**Funn:** Gjennom det meste av historien ligger gjennomsnittet stabilt høyt på den autoritære siden. Dette gir
mening siden de fleste historiske personer i datasettet er ledere, konger og keisere i autoritære samfunn.
Etter 1600-tallet faller kurven bratt mot libertære verdier, og moderne personer i datasettet har en mer
frihetlig, individualistisk og moderne samfunnsforståelse.

In [71]:
PC_ITEMS = ECON_LEFT + ECON_RIGHT + SOC_LIBERTARIAN + SOC_AUTHORITARIAN
order = ["strongly disagree", "disagree", "agree", "strongly agree"]
# AI til hjelp med å gjøre visualiseringen estetisk fin
pc_share = (train_df[PC_ITEMS].rename(columns=lambda c: c.split(".")[1])
            .melt(var_name="utsagn", value_name="svar").dropna()
            .groupby("utsagn")["svar"].value_counts(normalize=True).mul(100).rename("andel").reset_index())
fig = px.bar(pc_share, y="utsagn", x="andel", color="svar", orientation="h", category_orders={"svar": order},
             color_discrete_sequence=["#B2182B", "#EF8A62", "#67A9CF", "#2166AC"], height=500,
             title="Svar på de 12 politiske utsagnene (treningsdata)",
             labels={"utsagn": "utsagn", "andel": "andel personer (%)", "svar": "svar"})
fig.show()

**Funn:** Offentlig eide tjenester (87 % enig) og personlig ansvar (76 %) får bred støtte samtidig, selv
om de trekker i hver sin retning på den økonomiske aksen. Legalisering av narkotika er det eneste utsagnet et
klart flertall er uenig i (31 % enig). Svarene på enkeltutsagnene er altså mer nyanserte enn de to aksene alene
viser.

### 4.6 Tekstmønstre

Undersøker om modellen gjenbruker de samme svarene på tvers av ulike personer i fritekstfeltene.

In [72]:
top_books = train_df["personal.favorite_book"].value_counts().head(10).sort_values()
fig = px.bar(x=top_books.values, y=top_books.index, orientation="h",
             labels={"x": "antall personer", "y": "svar"},
             title="Mest gjenbrukte favorite_book-svar (treningsdata)")
fig.show()

top_animals = train_df["personal.favorite_animal"].value_counts().head(10).sort_values()
fig = px.bar(x=top_animals.values, y=top_animals.index, orientation="h",
             labels={"x": "antall personer", "y": "svar"},
             title="Mest gjenbrukte favorite_animal-svar (treningsdata)",)
fig.show()

**Funn**: Av 8000 er det 565 stykker som har "The Art of War" som favoritt, enda mer ekstremt har 1465 stykker "Falcon" som favorittdyr. Virker ikke som tydelig variasjon, av 8000 klumper det seg for mye rundt ett svar. Modellen kan trolig ha "default" svar som den velger på personaer når den er usikker.

### 4.7 Fasitkolonnene: fødeland og levealder

In [73]:
# Fasitkolonnene: fødeland og levealder
top_countries = train_df["born_country"].value_counts().head(15).sort_values()
fig = px.bar(x=top_countries.values, y=top_countries.index, orientation="h",
             title="De 15 vanligste fødelandene (treningsdata)",
             labels={"x": "antall personer", "y": "fødeland"})
fig.show()

life = real_train.dropna(subset=["born", "died"]).assign(levealder=lambda d: d["died"] - d["born"])
fig = px.histogram(life, x="levealder", nbins=80,
                   title="Levealder (died − born) for ekte personer (treningsdata)",
                   labels={"levealder": "levealder (år)"})
fig.update_yaxes(title_text="antall personer")
fig.show()
print("Ekte personer med negativ levealder:", (life["levealder"] < 0).sum())

Ekte personer med negativ levealder: 31


**Funn**: Datasettet er sterkt dominert av vestlige land. De fem vanligste fødelandene (USA, Frankrike, Storbritannia, Tyskland og Italia) står for nesten halvparten av personene med kjent fødeland, selv om det finnes 362 ulike land. Bare Kina, Japan, Egypt og India kommer inn på topp 15 fra utenfor Europa og Nord-Amerika. Dette er Pantheons skjevhet, ikke et utvalg vi har gjort. Modellene i oppgave A og B lærer derfor mest om vestlig historie, og funnene om hva språkmodellen «tror» gjelder først og fremst vestlige personer.

Levealderen er rimelig for de fleste ekte personer (median 66 år), men fasiten har tydelige feil. 31 personer i treningsdata dør før de er født (f.eks. Menkaure, født −2001 og død −2503), som er umulig uansett om årene er før eller etter Kristus. 27 personer lever over 120 år, opptil 2 869 år. Feilene gjelder nesten bare personer fra oldtiden, der årstallene er usikre eller anslått. Siden born er målvariabelen i oppgave A, er fasiten minst pålitelig akkurat der modellen bommer mest. Noe av feilen for oldtidspersoner skyldes altså usikker fasit, ikke bare modellen.

## 5. Variabelutvinning og pipeline (felles for oppgave A og B)
Begge oppgavene bruker **bare de 62 svarkolonnene** som input. Fasitkolonnene (`died`, `occupation`, `name`,
`type`, `gender` …) brukes aldri som prediktorer; `type` brukes bare til å evaluere ekte og fiktive hver for seg.

Svarkolonnene deles i grupper som får hver sin behandling:

| gruppe | behandling |
|---|---|
| tall (Big Five-ledd, prosenter, IQ, høyde, vekt …) | median-imputering + manglende-indikator + standardisering |
| politisk kompass | ordinal koding −2, −1, 1, 2 (som i datakatalogen) |
| moralske vekslingskurser | `log10`, siden de spenner over mange størrelsesordener |
| kategoriske svar | one-hot, med `"mangler"` som egen kategori |
| korte fritekstsvar (bok, maleri, dyr …) | små bokstaver, one-hot av svar som forekommer minst `min_frequency` ganger |
| setninger (Bergen, data science-fakta) | ordtelling med `CountVectorizer` (ord som står i minst 50 setninger) |
| nye variabler | Big Five-skårer, kompassakser, BMI, antall manglende svar, antall ord i setningene |

All behandling skjer inne i én pipeline (`FunctionTransformer` + `ColumnTransformer`), så medianer,
standardisering og hvilke tekstsvar som er vanlige nok, bare tilpasses på treningsdata.

In [74]:
PC_COLS  = [c for c in ANSWER_COLS if c.startswith("political_compass.")]
LOG_COLS = [c for c in ANSWER_COLS if c.startswith("moral_exchange_rates.")] + ["values.random_people_per_me"]
CATEGORY_COLS = [
    "values.friend_crime_testify", "values.child_quality_most_important", "values.free_will",
    "values.wants_to_live_forever", "personal.favorite_color", "personal.legacy_or_happiness",
    "personal.favorite_season",
]
TEXT_COLS = [
    "personal.favorite_person", "personal.favorite_painting", "personal.favorite_book",
    "personal.favorite_animal", "personal.last_meal",
]
SENTENCE_COLS = ["personal.favorite_thing_about_bergen", "personal.favorite_data_science_fact"]
NUMERIC_COLS = [c for c in ANSWER_COLS if c not in PC_COLS + LOG_COLS + CATEGORY_COLS + TEXT_COLS + SENTENCE_COLS]
assert len(NUMERIC_COLS + PC_COLS + LOG_COLS + CATEGORY_COLS + TEXT_COLS + SENTENCE_COLS) == 62   # alle kolonnene brukes


# Skåringen fra seksjon 2 gjenbrukes: score_big_five, score_political_compass, BIG_FIVE_TRAITS,
# PC_MAP og kompass-listene (ECON_LEFT osv.).

# Nye variabler
ENG_COLS = ([f"big_five_{t}" for t in BIG_FIVE_TRAITS] +
            ["econ_left_right", "soc_libertarian_authoritarian",
             "bmi", "n_missing", "words.bergen", "words.ds_fact"])

def prepare(X, engineered=True):
    X = X.copy()
    n_missing = X[ANSWER_COLS].isna().sum(axis=1) 
    if engineered:
        # Skåring fra del 1 — må skje før kompass svarene gjøres om til tall under,
        # fordi score_political_compass selv gjør tekst om til tall
        X = pd.concat([X, score_big_five(X), score_political_compass(X)], axis=1)
    for c in PC_COLS:
        X[c] = X[c].map(PC_MAP)
    for c in LOG_COLS:
        X[c] = np.log10(X[c])
    X["values.wants_to_live_forever"] = X["values.wants_to_live_forever"].map(
            lambda v: str(v) if pd.notna(v) else np.nan)      # True/False som tekst, NaN beholdes som manglende
    for c in TEXT_COLS:
        X[c] = X[c].str.lower().str.strip() 
    if engineered:
        X["bmi"] = X["personal.weight_kg"] / (X["personal.height_cm"] / 100) ** 2
        X["n_missing"] = n_missing
        X["words.bergen"] = X["personal.favorite_thing_about_bergen"].str.split().str.len()
        X["words.ds_fact"] = X["personal.favorite_data_science_fact"].str.split().str.len()
    for c in SENTENCE_COLS:
        X[c] = X[c].fillna("")
    return X

# Standardoppsett: alle variabelgrupper med. Variabelutvalget slår grupper av og på. AI hjalp med cfg
DEFAULT_CFG = dict(text=True, sentences=True, engineered=True, indicators=True, min_frequency=20)

def make_preprocessor(cfg):
    num_cols = NUMERIC_COLS + PC_COLS + LOG_COLS + (ENG_COLS if cfg["engineered"] else [])
    transformers = [
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=cfg["indicators"]), StandardScaler()), num_cols),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="mangler"), OneHotEncoder(handle_unknown="ignore")), CATEGORY_COLS)
        ]
    if cfg["text"]:
        transformers.append(
            ("txt", make_pipeline(SimpleImputer(strategy="constant", fill_value="mangler"), OneHotEncoder(min_frequency=cfg["min_frequency"], handle_unknown="infrequent_if_exist")), TEXT_COLS))
    if cfg["sentences"]:
        for c in SENTENCE_COLS:   # kolonnenavn som streng (ikke liste) gir CountVectorizer én tekstkolonne. CountVectorizer er AI sitt forslag
            
            # Gjør tekst om til tall ved å lage én kolonne per ord, 1 hvis ordet er i setningen og 0 ellers,
            # tar bare med ord som står i minst 50 setninger, stop_words="english" fjerner småord som "the"
            transformers.append((c.split(".")[1], CountVectorizer(binary=True, min_df=50, stop_words="english"), c))
    
    # Gir hver gruppe kolonner sin egen behandling og limer resultatene sammen,
    # Tallkolonnene får median og skalering, kategoriene får one-hot, og så videre.
    return ColumnTransformer(transformers)   # kolonner som ikke er nevnt, fjernes. ColumnTransformer er AI sitt forslag.

def make_model(regressor, cfg=None):
    cfg = {**DEFAULT_CFG, **(cfg or {})} # AI sin kortversjon, oppdaterer cfg med hva som ble tatt inn som argument
    
    # FunctionTransformer pakker inn prepare-funksjonen som et steg i pipelinen,
    # slik at pipelinen tar imot rådata direkte og alle transformasjoner skjer inne i den
    return make_pipeline(FunctionTransformer(prepare, kw_args={"engineered": cfg["engineered"]}), make_preprocessor(cfg), regressor)
                                 
# steg 0: FunctionTransformer(prepare), steg 1: ColumnTransformer(make_preprocessor), steg 2: modellen
# Antall variabler etter variabelutvinning (bare forbehandlingen tilpasses, på treningsdata)
pre = make_model(LinearRegression())[:-1].fit(split_train[ANSWER_COLS])
print("Antall variabler etter variabelutvinning:", len(pre[1].get_feature_names_out()))

Antall variabler etter variabelutvinning: 602


## 6. Oppgave A — Regresjon: når levde dette mennesket?
**Mål:** predikere `born` fra undersøkelsessvarene.

### 6.1 Data og målvariabel
Samme splitt som i seksjon 3. De 35 radene uten `born` kan verken brukes til trening eller evaluering, så de
fjernes **etter** splitten.

In [75]:
# Samme splitt som i seksjon 3. Rader uten fasit kan ikke brukes, så de fjernes etter splitten.
train_df, val_df, test_df = [d[d["born"].notna()] for d in (split_train, split_val, split_test)]

assert len(ANSWER_COLS) == 62
TARGET = "born"

X_train, y_train = train_df[ANSWER_COLS], train_df[TARGET]
X_val,   y_val   = val_df[ANSWER_COLS],   val_df[TARGET]
X_test,  y_test  = test_df[ANSWER_COLS],  test_df[TARGET]
print(f"train: {len(X_train)}  val: {len(X_val)}  test: {len(X_test)}")

assert len(train_df) + len(val_df) + len(test_df) == df["born"].notna().sum()
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == 62

print("Fornuftssjekker OK")

train: 7971  val: 998  test: 996
Fornuftssjekker OK


In [76]:
display(train_df.groupby("type")[TARGET].describe().round(0))

fig = px.histogram(train_df, x=TARGET, color="type", nbins=200, barmode="overlay",
                   title="Fødselsår (treningsdata)",
                   labels={"born": "fødselsår", "type": "type"})
fig.update_yaxes(title_text="antall personer")
fig.show()

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
fictional,764.0,853.0,2115.0,-10000.0,0.0,1905.0,1980.0,10190.0
real,7207.0,1508.0,755.0,-3500.0,1477.0,1856.0,1922.0,1999.0


**Funn:** fordelingen er svært skjev. Halvparten av personene er født etter ca. 1850, men halen går
tusenvis av år bakover, og noen fiktive figurer har absurde verdier (f.eks. −10 000 og 10 190).
Ingen modell kan forventes å treffe de ekstreme fiktive årstallene ut fra spørreskjemasvar.

### 6.2 Ytelsesmål

Jeg velger **MAE** (gjennomsnittlig absolutt feil, i år) som hovedmål:

- Den er lett å tolke: «modellen bommer i snitt med X år».
- Den er mindre følsom for uteliggere enn MSE/RMSE.

I tillegg rapporteres:
- **MedAE** (median absolutt feil) — hvor mye en typisk person bommes med,
- **MAE (ekte)** — MAE bare for ekte personer, siden fiktive fødselsår ofte er tull

In [77]:
results = []        # modellutvalg
models = {}         # navn -> funksjon som lager en ny, utrent modell (så vinneren kan bygges på nytt)

def evaluate(name, make, results_list=None):
    """Trener en ny modell fra make() på treningsdata og evaluerer på valideringsdata."""
    model = make()
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred = model.predict(X_val)
    real = (val_df["type"] == "real").to_numpy()    # type brukes kun til evaluering, ikke som input
    fict = (val_df["type"] == "fictional").to_numpy()
    row = {
        "modell": name,
        "MAE": mean_absolute_error(y_val, pred),
        "MedAE": median_absolute_error(y_val, pred),
        "MAE (ekte)": mean_absolute_error(y_val[real], pred[real]),
        "MAE (fiktive)": mean_absolute_error(y_val[fict], pred[fict]),
        "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
        "MAE (trening)": mean_absolute_error(y_train, pred_train)
    }
    (results if results_list is None else results_list).append(row)
    models[name] = make
    print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})

### 6.3 Grunnlinjer

`DummyRegressor` som alltid gjetter gjennomsnittet i treningsdata. Jeg tar også med
medianen, siden den passer bedre til en skjev fordeling og MAE. Alle andre modeller må slå disse.

In [78]:
# lambda er oppskrift på en modell
evaluate("Baseline: gjennomsnitt", lambda: DummyRegressor(strategy="mean"))
evaluate("Baseline: median", lambda: DummyRegressor(strategy="median"))

{'modell': 'Baseline: gjennomsnitt', 'MAE': 591.08, 'MedAE': 462.59, 'MAE (ekte)': 530.98, 'MAE (fiktive)': 1175.85, 'RMSE': np.float64(987.35), 'MAE (trening)': 615.56}
{'modell': 'Baseline: median', 'MAE': 453.4, 'MedAE': 91.0, 'MAE (ekte)': 392.37, 'MAE (fiktive)': 1047.35, 'RMSE': np.float64(1058.7), 'MAE (trening)': 484.58}


### 6.4 Lineær regresjon og log-transformasjon av målvariabelen

In [79]:
evaluate("Lineær regresjon", lambda cfg=None: make_model(LinearRegression(), cfg))

{'modell': 'Lineær regresjon', 'MAE': 685.08, 'MedAE': 256.44, 'MAE (ekte)': 332.05, 'MAE (fiktive)': 4120.45, 'RMSE': np.float64(9478.02), 'MAE (trening)': 389.09}


Lineær regresjon minimerer kvadrert feil, så den prøver hardt å treffe de få personene født tusenvis
av år f.Kr. og de absurde fiktive årstallene. Det ødelegger prediksjonene for alle de andre.

**Transformasjon:** i stedet for å predikere årstallet direkte, predikerer modellen «log-alder»

z = log(2030 - born)

dette setter en øvre grense på 2025 for de fiktive som er født i fremtiden, siden vi ikke kan ta log av et negativt tall. De fleste personene er født i nyere tid, mens noen få er født tusenvis av år tilbake. Uten transformasjon prøver modellen hardt å treffe de få ekstreme, og bommer på alle de andre. Log-alder hindrer dette ved å komprimere. I år er −10 000 mer enn 200 ganger lenger unna 1950 enn 1900. I log-skala er avstanden bare cirka 10 ganger større

In [80]:
REF_YEAR = 2030

def to_log_age(y):
    return np.log(REF_YEAR - np.clip(y, None, 2025))

def from_log_age(z):
    return REF_YEAR - np.exp(z)

def log_age(model):
    # transformerer målvariabelen før trening og transformerer prediksjonen tilbake etterpå,
    # for verdier over 2025 vil ikke den kunne reverseres og hente opprinnelig verdi
    return TransformedTargetRegressor(regressor=model, func=to_log_age, inverse_func=from_log_age, check_inverse=False)

fig = px.histogram(x=to_log_age(y_train), nbins=100, title="log(2030 - born) i treningsdata", labels={"x": "log-alder"})
fig.update_yaxes(title_text="antall personer")
fig.show()

evaluate("Lineær regresjon, log-alder", lambda: log_age(make_model(LinearRegression(), cfg=None)))

{'modell': 'Lineær regresjon, log-alder', 'MAE': 300.75, 'MedAE': 75.5, 'MAE (ekte)': 247.5, 'MAE (fiktive)': 819.0, 'RMSE': np.float64(769.53), 'MAE (trening)': 315.21}


Transformasjonen brukes for alle modellene videre.

### 6.5 Andre modeller
Hyperparameterne velges på valideringsdata.

**Lasso** straffer store koeffisienter og kan sette uviktige variabler til null; `alpha` styrer hvor hardt.

In [81]:
for alpha in [0.00003, 0.0001, 0.0003, 0.001, 0.003, 0.01]:
    evaluate(f"Lasso, log-alder, alpha={alpha}:", lambda cfg=None, a=alpha: log_age(make_model(Lasso(alpha=a, max_iter=10_000), cfg)))

{'modell': 'Lasso, log-alder, alpha=3e-05:', 'MAE': 300.03, 'MedAE': 74.66, 'MAE (ekte)': 246.8, 'MAE (fiktive)': 818.05, 'RMSE': np.float64(769.37), 'MAE (trening)': 314.78}
{'modell': 'Lasso, log-alder, alpha=0.0001:', 'MAE': 298.65, 'MedAE': 73.71, 'MAE (ekte)': 245.29, 'MAE (fiktive)': 817.9, 'RMSE': np.float64(769.17), 'MAE (trening)': 314.42}
{'modell': 'Lasso, log-alder, alpha=0.0003:', 'MAE': 296.58, 'MedAE': 73.32, 'MAE (ekte)': 242.97, 'MAE (fiktive)': 818.34, 'RMSE': np.float64(769.26), 'MAE (trening)': 314.66}
{'modell': 'Lasso, log-alder, alpha=0.001:', 'MAE': 297.63, 'MedAE': 73.79, 'MAE (ekte)': 242.06, 'MAE (fiktive)': 838.36, 'RMSE': np.float64(774.99), 'MAE (trening)': 319.41}
{'modell': 'Lasso, log-alder, alpha=0.003:', 'MAE': 313.09, 'MedAE': 78.14, 'MAE (ekte)': 253.43, 'MAE (fiktive)': 893.67, 'RMSE': np.float64(818.28), 'MAE (trening)': 336.45}
{'modell': 'Lasso, log-alder, alpha=0.01:', 'MAE': 337.62, 'MedAE': 87.75, 'MAE (ekte)': 282.09, 'MAE (fiktive)': 877.96

**k nærmeste naboer:** prediksjonen er gjennomsnittet av log-alderen til de k nærmeste personene i treningsdata.

In [82]:
for k in [1, 5, 10, 20, 50]:
    evaluate(f"kNN, log-alder, k={k}:", lambda cfg=None, k=k: log_age(make_model(KNeighborsRegressor(n_neighbors=k), cfg)))

{'modell': 'kNN, log-alder, k=1:', 'MAE': 432.18, 'MedAE': 65.0, 'MAE (ekte)': 355.41, 'MAE (fiktive)': 1179.23, 'RMSE': np.float64(1041.3), 'MAE (trening)': 2.82}
{'modell': 'kNN, log-alder, k=5:', 'MAE': 329.06, 'MedAE': 78.8, 'MAE (ekte)': 270.0, 'MAE (fiktive)': 903.75, 'RMSE': np.float64(832.03), 'MAE (trening)': 300.93}
{'modell': 'kNN, log-alder, k=10:', 'MAE': 340.75, 'MedAE': 78.8, 'MAE (ekte)': 274.88, 'MAE (fiktive)': 981.8, 'RMSE': np.float64(871.3), 'MAE (trening)': 329.8}
{'modell': 'kNN, log-alder, k=20:', 'MAE': 339.96, 'MedAE': 79.97, 'MAE (ekte)': 273.78, 'MAE (fiktive)': 983.91, 'RMSE': np.float64(886.92), 'MAE (trening)': 351.93}
{'modell': 'kNN, log-alder, k=50:', 'MAE': 350.8, 'MedAE': 83.43, 'MAE (ekte)': 282.42, 'MAE (fiktive)': 1016.2, 'RMSE': np.float64(925.74), 'MAE (trening)': 375.53}


**Beslutningstre:** `DecisionTreeRegressor` deler dataene med en serie ja/nei spørsmål og predikerer gjennomsnittet i hvert blad. Et dypt tre overtilpasser, så `max_depth` velges på valideringsdata.

In [83]:
for depth in [1, 2, 3, 5, 10, 15, None]:
    for min_leaf in [1, 5, 20]:
        evaluate(f"Beslutningstre, log-alder, max_depth={depth}, min_samples_leaf={min_leaf}", 
            lambda cfg=None, d=depth, min_leaf=min_leaf: log_age(make_model(DecisionTreeRegressor(max_depth=d, min_samples_leaf=min_leaf, random_state=SEED), cfg)))

{'modell': 'Beslutningstre, log-alder, max_depth=1, min_samples_leaf=1', 'MAE': 416.89, 'MedAE': 100.5, 'MAE (ekte)': 352.18, 'MAE (fiktive)': 1046.5, 'RMSE': np.float64(998.17), 'MAE (trening)': 450.57}
{'modell': 'Beslutningstre, log-alder, max_depth=1, min_samples_leaf=5', 'MAE': 416.89, 'MedAE': 100.5, 'MAE (ekte)': 352.18, 'MAE (fiktive)': 1046.5, 'RMSE': np.float64(998.17), 'MAE (trening)': 450.57}
{'modell': 'Beslutningstre, log-alder, max_depth=1, min_samples_leaf=20', 'MAE': 416.89, 'MedAE': 100.5, 'MAE (ekte)': 352.18, 'MAE (fiktive)': 1046.5, 'RMSE': np.float64(998.17), 'MAE (trening)': 450.57}
{'modell': 'Beslutningstre, log-alder, max_depth=2, min_samples_leaf=1', 'MAE': 403.37, 'MedAE': 99.21, 'MAE (ekte)': 337.97, 'MAE (fiktive)': 1039.77, 'RMSE': np.float64(982.35), 'MAE (trening)': 436.99}
{'modell': 'Beslutningstre, log-alder, max_depth=2, min_samples_leaf=5', 'MAE': 403.37, 'MedAE': 99.21, 'MAE (ekte)': 337.97, 'MAE (fiktive)': 1039.77, 'RMSE': np.float64(982.35), 'M

**Tilfeldig skog:** `RandomForestRegressor` trener mange trær på tilfeldige utvalg av rader og variabler og tar gjennomsnittet,
som reduserer overtilpasningen til ett enkelt tre. Jeg tester `max_features` (andel variabler hvert splitt
kan velge mellom) og `min_samples_leaf`(minste antall personer på hvert blad)

In [84]:
for mf in [0.1, 0.3, 1.0]:
    for min_leaf in [1, 5, 20]:
        evaluate(f"Tilfeldig skog, log-alder, max_features={mf}, min_samples_leaf={min_leaf}",
             lambda cfg=None, mf=mf, min_leaf=min_leaf: log_age(make_model(RandomForestRegressor(
                 n_estimators=200, max_features=mf, min_samples_leaf=min_leaf, n_jobs=-1, random_state=SEED), cfg)))

{'modell': 'Tilfeldig skog, log-alder, max_features=0.1, min_samples_leaf=1', 'MAE': 308.75, 'MedAE': 64.44, 'MAE (ekte)': 241.34, 'MAE (fiktive)': 964.74, 'RMSE': np.float64(865.83), 'MAE (trening)': 169.95}
{'modell': 'Tilfeldig skog, log-alder, max_features=0.1, min_samples_leaf=5', 'MAE': 321.66, 'MedAE': 68.75, 'MAE (ekte)': 253.79, 'MAE (fiktive)': 982.14, 'RMSE': np.float64(890.48), 'MAE (trening)': 292.56}
{'modell': 'Tilfeldig skog, log-alder, max_features=0.1, min_samples_leaf=20', 'MAE': 342.45, 'MedAE': 79.15, 'MAE (ekte)': 274.12, 'MAE (fiktive)': 1007.38, 'RMSE': np.float64(925.54), 'MAE (trening)': 360.88}
{'modell': 'Tilfeldig skog, log-alder, max_features=0.3, min_samples_leaf=1', 'MAE': 299.43, 'MedAE': 61.61, 'MAE (ekte)': 230.63, 'MAE (fiktive)': 968.86, 'RMSE': np.float64(851.73), 'MAE (trening)': 160.79}
{'modell': 'Tilfeldig skog, log-alder, max_features=0.3, min_samples_leaf=5', 'MAE': 311.34, 'MedAE': 66.97, 'MAE (ekte)': 243.69, 'MAE (fiktive)': 969.6, 'RMSE':

### 6.6 Sammenligning av modellene

In [85]:
res = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)

# Alle forsøkene i en tabell
display(res.round(2))

# Beste variant av hver modelltype (navnet før første komma: "Lasso", "kNN", "Tilfeldig skog", ...)
res["modelltype"] = res["modell"].str.split(",").str[0]
best_per_type = res.drop_duplicates("modelltype").iloc[::-1]   # snu, så den beste havner øverst

fig = go.Figure([
    go.Bar(name="trening", y=best_per_type["modell"], x=best_per_type["MAE (trening)"], orientation="h"),
    go.Bar(name="validering", y=best_per_type["modell"], x=best_per_type["MAE"], orientation="h"),
])
fig.update_layout(barmode="group", height=550, xaxis_title="MAE (år)", yaxis_title="modell",
                  title="Beste variant av hver modelltype: trening vs validering (oppgave A)")
fig.show()

best_name = res.loc[0, "modell"]
# oppskriften til den beste modellen lagret i models dictionary
best_make = models[best_name]
print("Beste modell på valideringsdata:", best_name)

,modell,MAE,MedAE,MAE (ekte),MAE (fiktive),RMSE,MAE (trening)
0,"Lasso, log-alder, alpha=0.0003:",296.58,73.32,242.97,818.34,769.26,314.66
1,"Lasso, log-alder, alpha=0.001:",297.63,73.79,242.06,838.36,774.99,319.41
2,"Tilfeldig skog, log-alder, max_features=1.0, m...",297.82,62.40,227.97,977.51,848.15,153.81
3,"Lasso, log-alder, alpha=0.0001:",298.65,73.71,245.29,817.90,769.17,314.42
4,"Tilfeldig skog, log-alder, max_features=0.3, m...",299.43,61.61,230.63,968.86,851.73,160.79
5,"Lasso, log-alder, alpha=3e-05:",300.03,74.66,246.80,818.05,769.37,314.78
6,"Lineær regresjon, log-alder",300.75,75.50,247.50,819.00,769.53,315.21
7,"Tilfeldig skog, log-alder, max_features=1.0, m...",307.78,62.82,238.51,981.85,861.69,234.90
8,"Tilfeldig skog, log-alder, max_features=0.1, m...",308.75,64.44,241.34,964.74,865.83,169.95
9,"Tilfeldig skog, log-alder, max_features=0.3, m...",311.34,66.97,243.69,969.60,864.06,260.27


Beste modell på valideringsdata: Lasso, log-alder, alpha=0.0003:


Ser at alle som er har mindre feil på treningsdata enn valideringsdata er overtilpasset. Trodde kanskje tilfeldig skog skulle være bra, men er veldig overtilpasset. Ser også at lasso med alpha = 0.0003 og 0.001 er to bra modeller, men den med alpha = 0.0003 er akkurat litt bedre med mindre feil på både trenings- og valideringsdata.

### 6.7 Variabelutvalg på valideringsdata

Sjekker om alle variabelgruppene hjelper på prediksjonen eller om noe bare er støy. Bruker den beste modellen fra forrige seksjon og fjerner én gruppe om gangen og sammenligner MAE

In [86]:
VARIANTS = {
    "uten fritekst-svar (one-hot)": {"text": False},
    "uten ord fra setningene":      {"sentences": False},
    "uten nye variabler":           {"engineered": False},
    "uten manglende indikatorer":   {"indicators": False},
    "fritekst min_frequency=2":    {"min_frequency": 2},
    "fritekst min_frequency=3":    {"min_frequency": 3},
    "fritekst min_frequency=5":    {"min_frequency": 5},
    "fritekst min_frequency=10":    {"min_frequency": 10},
    "fritekst min_frequency=20":    {"min_frequency": 20},
    "fritekst min_frequency=50":    {"min_frequency": 50},
}

fs_results = []
for name, change in VARIANTS.items():
    evaluate(name, lambda: best_make(change), results_list=fs_results)

fs = pd.DataFrame(fs_results).sort_values("MAE").reset_index(drop=True)
display(fs.round(2))

fig = go.Figure([
    go.Bar(name="trening", x=fs["modell"], y=fs["MAE (trening)"]),
    go.Bar(name="validering", x=fs["modell"], y=fs["MAE"]),
])
fig.update_layout(barmode="group", height=800, yaxis_title="MAE (år)", yaxis_range=[270, 360], title="Variabelutvalg: trenings- vs valideringsfeil (oppgave A)", xaxis_title="variant")
fig.show()

best_variant = fs.loc[0, "modell"]
final_cfg = VARIANTS[best_variant]

{'modell': 'uten fritekst-svar (one-hot)', 'MAE': 342.78, 'MedAE': 86.65, 'MAE (ekte)': 280.14, 'MAE (fiktive)': 952.32, 'RMSE': np.float64(889.28), 'MAE (trening)': 355.18}
{'modell': 'uten ord fra setningene', 'MAE': 310.61, 'MedAE': 75.75, 'MAE (ekte)': 250.25, 'MAE (fiktive)': 897.94, 'RMSE': np.float64(813.94), 'MAE (trening)': 331.69}
{'modell': 'uten nye variabler', 'MAE': 299.17, 'MedAE': 73.11, 'MAE (ekte)': 243.07, 'MAE (fiktive)': 845.06, 'RMSE': np.float64(772.11), 'MAE (trening)': 311.8}
{'modell': 'uten manglende indikatorer', 'MAE': 298.62, 'MedAE': 73.74, 'MAE (ekte)': 243.34, 'MAE (fiktive)': 836.59, 'RMSE': np.float64(771.27), 'MAE (trening)': 312.73}
{'modell': 'fritekst min_frequency=2', 'MAE': 285.02, 'MedAE': 69.57, 'MAE (ekte)': 229.45, 'MAE (fiktive)': 825.85, 'RMSE': np.float64(758.75), 'MAE (trening)': 282.79}
{'modell': 'fritekst min_frequency=3', 'MAE': 283.55, 'MedAE': 68.66, 'MAE (ekte)': 228.61, 'MAE (fiktive)': 818.19, 'RMSE': np.float64(753.0), 'MAE (tr

,modell,MAE,MedAE,MAE (ekte),MAE (fiktive),RMSE,MAE (trening)
0,fritekst min_frequency=3,283.55,68.66,228.61,818.19,753.00,285.76
1,fritekst min_frequency=5,284.93,70.09,229.97,819.72,754.72,291.12
2,fritekst min_frequency=2,285.02,69.57,229.45,825.85,758.75,282.79
3,fritekst min_frequency=10,292.27,71.35,236.94,830.67,762.93,304.73
4,fritekst min_frequency=20,296.58,73.32,242.97,818.34,769.26,314.66
5,uten manglende indikatorer,298.62,73.74,243.34,836.59,771.27,312.73
6,uten nye variabler,299.17,73.11,243.07,845.06,772.11,311.80
7,fritekst min_frequency=50,303.95,72.94,250.42,824.83,782.39,324.32
8,uten ord fra setningene,310.61,75.75,250.25,897.94,813.94,331.69
9,uten fritekst-svar (one-hot),342.78,86.65,280.14,952.32,889.28,355.18


**Funn**: Ved å senke minimum samme svar på fritekst feltene ser jeg at feilen synker helt fram til min_frequency = 3, når den blir lik 2 synker treningsfeilen, men valideringsfeilen øker som er et tegn på at modellen er overtilpasset 

### 6.8 Endelig modell, én evaluering på testdata

In [87]:
final_model = best_make(final_cfg)
# Tren modellen på slått sammen data
X_trval = pd.concat([X_train, X_val])
y_trval = pd.concat([y_train, y_val])
final_model.fit(X_trval, y_trval)
test_pred = final_model.predict(X_test)

real_test = (test_df["type"] == "real").to_numpy()
test_scores = pd.Series({
    "MAE": mean_absolute_error(y_test, test_pred),
    "MedAE": median_absolute_error(y_test, test_pred),
    "MAE (ekte)": mean_absolute_error(y_test[real_test], test_pred[real_test]),
    "RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
}, name=f"{best_name} | {best_variant}")
display(test_scores.round(2).to_frame())

,"Lasso, log-alder, alpha=0.0003: | fritekst min_frequency=3"
MAE,351.34
MedAE,74.79
MAE (ekte),265.42
RMSE,781.40


In [88]:
plot_df = test_df[["name", "type", "occupation_category", TARGET]].copy()
plot_df["pred"] = test_pred
plot_df["abs_err"] = (plot_df["pred"] - plot_df[TARGET]).abs()

lo, hi = -3000, 2100
fig = px.scatter(plot_df, x=TARGET, y="pred", color="type", hover_name="name", opacity=0.5,
                title="Predikert vs. faktisk fødselsår (testdata)", range_x=[lo, hi], range_y=[lo, hi],
                labels={TARGET: "faktisk fødselsår", "pred": "predikert fødselsår", "type": "type"})

# AI hjelp med plot
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="perfekt", line=dict(dash="dash")))
fig.show()

In [89]:
display(plot_df.groupby("type")["abs_err"].agg(["count", "mean", "median"]).round(0))

# De 10 største feilene
display(plot_df.sort_values("abs_err", ascending=False).head(10))

,count,mean,median
type,,,
fictional,108,1058.0,245.0
real,888,265.0,68.0


,name,type,occupation_category,born,pred,abs_err
9907,Anguirus,fictional,sport,-5000.0,286.081805,5286.081805
9416,Gríma Wormtongue,fictional,politics & rule,-2970.0,1740.037800,4710.037800
9186,TARDIS,fictional,other,0.0,-4586.832150,4586.832150
9155,Gimli,fictional,military,-2941.0,1637.744596,4578.744596
3102,Wazner,real,politics & rule,-3500.0,1006.200359,4506.200359
9720,Beorn,fictional,politics & rule,-2950.0,1479.485389,4429.485389
9380,Bhrigu,fictional,philosophy & scholarship,-3000.0,1382.347432,4382.347432
6047,Den,real,philosophy & scholarship,-3000.0,1267.666441,4267.666441
9304,Adapa,fictional,philosophy & scholarship,-3000.0,1259.440628,4259.440628
9291,Fereydun,fictional,politics & rule,-3000.0,1037.792638,4037.792638


Fiktive personer bommes med langt mer enn ekte. De største feilene er nesten bare fiktive figurer og personer fra antikken. Det forklarer hvorfor MAE er mye høyere enn MedAE: noen få ekstreme personer trekker gjennomsnittet opp. Den endelige modellen fikk MAE 351 år på testdata, mot 284 på validering. MedAE var nesten uendret (74 mot 69 år). Modellen generaliserer altså godt for den typiske personen, og den høyere MAE skyldes noen få personer med ekstreme fødselsår, hovedsakelig fiktive figurer og personer fra antikken. Halvparten av testpersonene plasseres innenfor 74 år av riktig fødselsår.

## 7. Oppgave B — Klassifikasjon: hva drev denne personen med?
**Mål:** predikere `occupation_category` (10 klasser) fra undersøkelsessvarene, med samme splitt og samme
pipeline som i oppgave A.

### 7.1 Data og målvariabel

In [90]:
TARGET_B = "occupation_category"

# Samme splitt som i del 1 og oppgave A; rader uten yrkeskategori fjernes etter splitten.
train_df_b, val_df_b, test_df_b = [d[d[TARGET_B].notna()] for d in (split_train, split_val, split_test)]

X_train_b, y_train_b = train_df_b[ANSWER_COLS], train_df_b[TARGET_B]
X_val_b,   y_val_b   = val_df_b[ANSWER_COLS],   val_df_b[TARGET_B]
X_test_b,  y_test_b  = test_df_b[ANSWER_COLS],  test_df_b[TARGET_B]
print(f"train: {len(X_train_b)}  val: {len(X_val_b)}  test: {len(X_test_b)}")

CLASSES = y_train_b.value_counts().index.tolist()   # sortert fra største til minste klasse

train: 7970  val: 998  test: 996


In [91]:
counts_b = y_train_b.value_counts()
display(pd.DataFrame({"antall": counts_b, "prosent": (counts_b / len(y_train_b) * 100).round(1)}))

fig = px.bar(x=counts_b.index, y=counts_b.values, title="Yrkeskategori i treningssettet",
             labels={"x": "yrkeskategori", "y": "antall personer"})
fig.show()

display(pd.crosstab(train_df_b[TARGET_B], train_df_b["type"]).loc[CLASSES])

,antall,prosent
occupation_category,,
politics & rule,2833,35.5
performing arts,1120,14.1
science & medicine,956,12.0
writing,695,8.7
philosophy & scholarship,517,6.5
other,509,6.4
visual arts & design,400,5.0
military,330,4.1
sport,311,3.9


type,fictional,real
occupation_category,,
politics & rule,116,2717
performing arts,88,1032
science & medicine,59,897
writing,35,660
philosophy & scholarship,29,488
other,189,320
visual arts & design,8,392
military,62,268
sport,56,255


**Funn:** Klassene er svært ubalanserte. `politics & rule` utgjør 35.5 % av treningsdata, mens den minste klassen,
`famous by association`, bare utgjør 3.8 %, altså omtrent 9 ganger færre. De tre største klassene utgjør til sammen 61 %.
En modell som alltid gjetter `politics & rule` vil derfor få 35.5 % riktig uten å ha lært noe, så nøyaktighet alene
blir misvisende. Jeg trenger et ytelsesmål der alle klassene teller likt.

De fiktive personene er ikke jevnt fordelt. Totalt er ca. 10 % fiktive, men i `famous by association` er 41 % fiktive
og i `other` 37 %, mot bare 4 % i `politics & rule`. Disse to klassene er også vage samlekategorier. Jeg forventer
derfor at de blir de vanskeligste å predikere, siden de er små, har mange fiktive og er uklart avgrenset.

### 7.2 Ytelsesmål

For ytelsesmål er det F1 score som velges som hovedmål på grunn av at hvis enten dekning eller presisjon er lav vil F1 bli lav. Den eneste måten å få høy F1 på er å både finne personene i klassen og ikke rope falsk alarm. F1 ignorerer også true negative som er smart her fordi for en liten klasse som famous by association er nesten alle personene `tn`, så et mål som tok med `tn` ville sett bra ut uansett. Fra: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html får jeg at 'macro' 'does not take label imbalance into account', som er akkurat det jeg vil her.

I tillegg rapporteres:
- **Nøyaktighet** (accuracy) — andelen som klassifiseres riktig,
- **makro-F1 (ekte)** — makro-F1 bare for ekte personer,
- **makro-F1 (trening)** — for å se overtilpasning.

In [92]:
results_b = []
models_b = {}

def evaluate_clf(name, make, results_list=None):
    model = make()
    model.fit(X_train_b, y_train_b)
    pred_train = model.predict(X_train_b)
    pred = model.predict(X_val_b)
    real = (val_df_b["type"] == "real").to_numpy()   # type brukes kun til evaluering, ikke som input
    row = {
        "modell": name,
        "makro-F1": f1_score(y_val_b, pred, average="macro"),
        "nøyaktighet": accuracy_score(y_val_b, pred),
        "makro-F1 (ekte)": f1_score(y_val_b[real], pred[real], average="macro"),
        "makro-F1 (trening)": f1_score(y_train_b, pred_train, average="macro"),
    }
    (results_b if results_list is None else results_list).append(row)
    models_b[name] = make
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in row.items()})

### 7.3 Grunnlinjer

`DummyClassifier` som alltid gjetter den vanligste klassen (`politics & rule`), og en som gjetter tilfeldig
med samme klassefordeling som i treningsdata. Alle andre modeller må slå disse.

In [93]:
evaluate_clf("Baseline: vanligste klasse", lambda: DummyClassifier(strategy="most_frequent"))
evaluate_clf("Baseline: tilfeldig (stratifisert)", lambda: DummyClassifier(strategy="stratified", random_state=SEED))

{'modell': 'Baseline: vanligste klasse', 'makro-F1': 0.052, 'nøyaktighet': 0.356, 'makro-F1 (ekte)': 0.056, 'makro-F1 (trening)': 0.052}
{'modell': 'Baseline: tilfeldig (stratifisert)', 'makro-F1': 0.091, 'nøyaktighet': 0.173, 'makro-F1 (ekte)': 0.087, 'makro-F1 (trening)': 0.098}


### 7.4 Modeller
**Logistisk regresjon:** `C` styrer regulariseringen (som `alpha` i Lasso, men omvendt, liten `C` = sterk straff).

In [94]:
for C in [0.01, 0.1, 1, 10]:
    for weight in [None, "balanced"]:
        evaluate_clf(f"Logistisk regresjon, C = {C}, class_weight={weight}", lambda cfg=None, c=C, cw=weight: make_model(LogisticRegression(max_iter=5000, C=c, class_weight=cw), cfg))

{'modell': 'Logistisk regresjon, C = 0.01, class_weight=None', 'makro-F1': 0.624, 'nøyaktighet': 0.741, 'makro-F1 (ekte)': 0.639, 'makro-F1 (trening)': 0.667}
{'modell': 'Logistisk regresjon, C = 0.01, class_weight=balanced', 'makro-F1': 0.666, 'nøyaktighet': 0.714, 'makro-F1 (ekte)': 0.687, 'makro-F1 (trening)': 0.683}
{'modell': 'Logistisk regresjon, C = 0.1, class_weight=None', 'makro-F1': 0.696, 'nøyaktighet': 0.786, 'makro-F1 (ekte)': 0.727, 'makro-F1 (trening)': 0.767}
{'modell': 'Logistisk regresjon, C = 0.1, class_weight=balanced', 'makro-F1': 0.694, 'nøyaktighet': 0.753, 'makro-F1 (ekte)': 0.721, 'makro-F1 (trening)': 0.763}
{'modell': 'Logistisk regresjon, C = 1, class_weight=None', 'makro-F1': 0.691, 'nøyaktighet': 0.781, 'makro-F1 (ekte)': 0.734, 'makro-F1 (trening)': 0.836}
{'modell': 'Logistisk regresjon, C = 1, class_weight=balanced', 'makro-F1': 0.683, 'nøyaktighet': 0.747, 'makro-F1 (ekte)': 0.721, 'makro-F1 (trening)': 0.834}
{'modell': 'Logistisk regresjon, C = 10, c

**k nærmeste naboer:**

In [95]:
for k in [1, 2, 3, 5, 10, 20, 50]:
    evaluate_clf(f"kNN, k = {k}", lambda cfg=None, k=k: make_model(KNeighborsClassifier(n_neighbors=k), cfg))

{'modell': 'kNN, k = 1', 'makro-F1': 0.545, 'nøyaktighet': 0.65, 'makro-F1 (ekte)': 0.575, 'makro-F1 (trening)': 1.0}
{'modell': 'kNN, k = 2', 'makro-F1': 0.53, 'nøyaktighet': 0.62, 'makro-F1 (ekte)': 0.561, 'makro-F1 (trening)': 0.778}
{'modell': 'kNN, k = 3', 'makro-F1': 0.563, 'nøyaktighet': 0.678, 'makro-F1 (ekte)': 0.607, 'makro-F1 (trening)': 0.763}
{'modell': 'kNN, k = 5', 'makro-F1': 0.573, 'nøyaktighet': 0.691, 'makro-F1 (ekte)': 0.61, 'makro-F1 (trening)': 0.719}
{'modell': 'kNN, k = 10', 'makro-F1': 0.564, 'nøyaktighet': 0.701, 'makro-F1 (ekte)': 0.594, 'makro-F1 (trening)': 0.65}
{'modell': 'kNN, k = 20', 'makro-F1': 0.551, 'nøyaktighet': 0.702, 'makro-F1 (ekte)': 0.582, 'makro-F1 (trening)': 0.592}
{'modell': 'kNN, k = 50', 'makro-F1': 0.527, 'nøyaktighet': 0.689, 'makro-F1 (ekte)': 0.547, 'makro-F1 (trening)': 0.529}


**Beslutningstre og tilfeldig skog:**

In [96]:
for depth in [1, 2, 3, 5, 10, 15, None]:
    for min_leaf in [1, 5, 20]:
        evaluate_clf(f"Beslutningstre, depth={depth}, min_samples_leaf={min_leaf}", lambda cfg=None, d=depth, min_leaf=min_leaf: make_model(DecisionTreeClassifier(max_depth=d, min_samples_leaf=min_leaf, random_state=SEED), cfg))

{'modell': 'Beslutningstre, depth=1, min_samples_leaf=1', 'makro-F1': 0.1, 'nøyaktighet': 0.368, 'makro-F1 (ekte)': 0.104, 'makro-F1 (trening)': 0.102}
{'modell': 'Beslutningstre, depth=1, min_samples_leaf=5', 'makro-F1': 0.1, 'nøyaktighet': 0.368, 'makro-F1 (ekte)': 0.104, 'makro-F1 (trening)': 0.102}
{'modell': 'Beslutningstre, depth=1, min_samples_leaf=20', 'makro-F1': 0.1, 'nøyaktighet': 0.368, 'makro-F1 (ekte)': 0.104, 'makro-F1 (trening)': 0.102}
{'modell': 'Beslutningstre, depth=2, min_samples_leaf=1', 'makro-F1': 0.119, 'nøyaktighet': 0.468, 'makro-F1 (ekte)': 0.125, 'makro-F1 (trening)': 0.119}
{'modell': 'Beslutningstre, depth=2, min_samples_leaf=5', 'makro-F1': 0.119, 'nøyaktighet': 0.468, 'makro-F1 (ekte)': 0.125, 'makro-F1 (trening)': 0.119}
{'modell': 'Beslutningstre, depth=2, min_samples_leaf=20', 'makro-F1': 0.119, 'nøyaktighet': 0.468, 'makro-F1 (ekte)': 0.125, 'makro-F1 (trening)': 0.119}
{'modell': 'Beslutningstre, depth=3, min_samples_leaf=1', 'makro-F1': 0.182, 'nø

In [97]:
for mf in [0.1, 0.3, 1.0]:
    for weight in [None, "balanced"]:
        for n in [50, 100, 200, 400]:
            evaluate_clf(f"Tilfeldig skog, max_features = {mf}, class_weight={weight}, antall trær={n}", lambda cfg=None, mf=mf, cw=weight, n=n: make_model(RandomForestClassifier(n_estimators=n, max_features=mf, class_weight=cw, n_jobs=-1, random_state=SEED), cfg))

{'modell': 'Tilfeldig skog, max_features = 0.1, class_weight=None, antall trær=50', 'makro-F1': 0.609, 'nøyaktighet': 0.729, 'makro-F1 (ekte)': 0.636, 'makro-F1 (trening)': 1.0}
{'modell': 'Tilfeldig skog, max_features = 0.1, class_weight=None, antall trær=100', 'makro-F1': 0.609, 'nøyaktighet': 0.73, 'makro-F1 (ekte)': 0.632, 'makro-F1 (trening)': 1.0}
{'modell': 'Tilfeldig skog, max_features = 0.1, class_weight=None, antall trær=200', 'makro-F1': 0.601, 'nøyaktighet': 0.729, 'makro-F1 (ekte)': 0.628, 'makro-F1 (trening)': 1.0}
{'modell': 'Tilfeldig skog, max_features = 0.1, class_weight=None, antall trær=400', 'makro-F1': 0.594, 'nøyaktighet': 0.727, 'makro-F1 (ekte)': 0.625, 'makro-F1 (trening)': 1.0}
{'modell': 'Tilfeldig skog, max_features = 0.1, class_weight=balanced, antall trær=50', 'makro-F1': 0.652, 'nøyaktighet': 0.726, 'makro-F1 (ekte)': 0.688, 'makro-F1 (trening)': 0.974}
{'modell': 'Tilfeldig skog, max_features = 0.1, class_weight=balanced, antall trær=100', 'makro-F1': 0

**SVM:**

In [98]:
for C in [1, 3, 10, 30]:
    for weight in [None, "balanced"]:
        evaluate_clf(f"SVM, C = {C}, class_weight={weight}", lambda cfg=None, c=C, cw=weight: make_model(SVC(C=c, class_weight=cw, random_state=SEED), cfg))

{'modell': 'SVM, C = 1, class_weight=None', 'makro-F1': 0.644, 'nøyaktighet': 0.763, 'makro-F1 (ekte)': 0.657, 'makro-F1 (trening)': 0.811}
{'modell': 'SVM, C = 1, class_weight=balanced', 'makro-F1': 0.703, 'nøyaktighet': 0.754, 'makro-F1 (ekte)': 0.726, 'makro-F1 (trening)': 0.835}
{'modell': 'SVM, C = 3, class_weight=None', 'makro-F1': 0.705, 'nøyaktighet': 0.787, 'makro-F1 (ekte)': 0.745, 'makro-F1 (trening)': 0.948}
{'modell': 'SVM, C = 3, class_weight=balanced', 'makro-F1': 0.726, 'nøyaktighet': 0.783, 'makro-F1 (ekte)': 0.766, 'makro-F1 (trening)': 0.935}
{'modell': 'SVM, C = 10, class_weight=None', 'makro-F1': 0.718, 'nøyaktighet': 0.791, 'makro-F1 (ekte)': 0.765, 'makro-F1 (trening)': 0.999}
{'modell': 'SVM, C = 10, class_weight=balanced', 'makro-F1': 0.725, 'nøyaktighet': 0.793, 'makro-F1 (ekte)': 0.771, 'makro-F1 (trening)': 0.989}
{'modell': 'SVM, C = 30, class_weight=None', 'makro-F1': 0.71, 'nøyaktighet': 0.786, 'makro-F1 (ekte)': 0.757, 'makro-F1 (trening)': 1.0}
{'modell

### 7.5 Sammenligning og valg av modell

In [99]:
res_b = pd.DataFrame(results_b).sort_values("makro-F1", ascending=False).reset_index(drop=True)

# Alle forsøkene i en tabell (alle radene vises)
pd.set_option("display.max_rows", 100)
display(res_b.round(3))

# Beste variant av hver modelltype 
res_b["modelltype"] = res_b["modell"].str.split(",").str[0]
best_per_type = res_b.drop_duplicates("modelltype").iloc[::-1]   # snu, så den beste havner øverst

fig = go.Figure([
    go.Bar(name="trening", y=best_per_type["modell"], x=best_per_type["makro-F1 (trening)"], orientation="h"),
    go.Bar(name="validering", y=best_per_type["modell"], x=best_per_type["makro-F1"], orientation="h"),
])
fig.update_layout(barmode="group", height=550, xaxis_title="makro-F1", yaxis_title="modell",
                  title="Beste variant av hver modelltype: trening vs validering (oppgave B)")
fig.show()

,modell,makro-F1,nøyaktighet,makro-F1 (ekte),makro-F1 (trening)
0,"SVM, C = 3, class_weight=balanced",0.726,0.783,0.766,0.935
1,"SVM, C = 10, class_weight=balanced",0.725,0.793,0.771,0.989
2,"SVM, C = 10, class_weight=None",0.718,0.791,0.765,0.999
3,"SVM, C = 30, class_weight=balanced",0.713,0.788,0.758,0.999
4,"SVM, C = 30, class_weight=None",0.710,0.786,0.757,1.000
5,"SVM, C = 3, class_weight=None",0.705,0.787,0.745,0.948
6,"SVM, C = 1, class_weight=balanced",0.703,0.754,0.726,0.835
7,"Logistisk regresjon, C = 0.1, class_weight=None",0.696,0.786,0.727,0.767
8,"Logistisk regresjon, C = 0.1, class_weight=bal...",0.694,0.753,0.721,0.763
9,"Logistisk regresjon, C = 1, class_weight=None",0.691,0.781,0.734,0.836


Jeg velger modellen med høyest makro-F1 på valideringsdata: **SVM med `C = 3` og `class_weight="balanced"`**.
Valideringsdataene er ikke brukt til å trene modellene, så valideringsskåren er det beste anslaget på hvordan
modellen vil gjøre det på nye personer. SVM har nesten perfekt skår på treningsdata, altså er den overtilpasset,
men den generaliserer likevel bedre enn de andre modellene. `class_weight="balanced"` gir personer i små klasser
større vekt i treningen, som passer med makro-F1, der alle klassene teller likt.

### 7.6 Variabelutvalg på valideringsdata
Samme varianter som i oppgave A, med den valgte modellen (`SVM`).

In [100]:
chosen_model_b = res_b.loc[0, "modell"]   # res_b er sortert etter makro-F1 på validering, høyest først
chosen_make_b = models_b[chosen_model_b]

VARIANTS_B = {
    "uten fritekst-svar (one-hot)": {"text": False},
    "uten ord fra setningene":      {"sentences": False},
    "uten nye variabler":           {"engineered": False},
    "uten manglende indikatorer":   {"indicators": False},
    "fritekst min_frequency=2":    {"min_frequency": 2},
    "fritekst min_frequency=3":    {"min_frequency": 3},
    "fritekst min_frequency=5":    {"min_frequency": 5},
    "fritekst min_frequency=10":    {"min_frequency": 10},
    "fritekst min_frequency=20":    {"min_frequency": 20},
    "fritekst min_frequency=50":    {"min_frequency": 50},
}

fs_results_b = []
for name, change in VARIANTS_B.items():
    evaluate_clf(name, lambda: chosen_make_b(change), results_list=fs_results_b)
    
fs_b = pd.DataFrame(fs_results_b).sort_values("makro-F1", ascending=False).reset_index(drop=True)
display(fs_b.round(3))

fig = go.Figure([
    go.Bar(name="trening", x=fs_b["modell"], y=fs_b["makro-F1 (trening)"]),
    go.Bar(name="validering", x=fs_b["modell"], y=fs_b["makro-F1"]),
])
fig.update_layout(barmode="group", height=800, yaxis_title="makro-F1", title="Variabelutvalg: trenings- vs valideringsskår (oppgave B)", xaxis_title="variant")
fig.show()

best_variant_b = fs_b.loc[0, "modell"]
final_cfg_b = VARIANTS_B[best_variant_b]
print("Beste variant:", best_variant_b)

{'modell': 'uten fritekst-svar (one-hot)', 'makro-F1': 0.682, 'nøyaktighet': 0.752, 'makro-F1 (ekte)': 0.713, 'makro-F1 (trening)': 0.912}
{'modell': 'uten ord fra setningene', 'makro-F1': 0.696, 'nøyaktighet': 0.762, 'makro-F1 (ekte)': 0.725, 'makro-F1 (trening)': 0.888}
{'modell': 'uten nye variabler', 'makro-F1': 0.728, 'nøyaktighet': 0.789, 'makro-F1 (ekte)': 0.773, 'makro-F1 (trening)': 0.942}
{'modell': 'uten manglende indikatorer', 'makro-F1': 0.727, 'nøyaktighet': 0.788, 'makro-F1 (ekte)': 0.769, 'makro-F1 (trening)': 0.948}
{'modell': 'fritekst min_frequency=2', 'makro-F1': 0.73, 'nøyaktighet': 0.786, 'makro-F1 (ekte)': 0.764, 'makro-F1 (trening)': 0.945}
{'modell': 'fritekst min_frequency=3', 'makro-F1': 0.724, 'nøyaktighet': 0.783, 'makro-F1 (ekte)': 0.763, 'makro-F1 (trening)': 0.944}
{'modell': 'fritekst min_frequency=5', 'makro-F1': 0.729, 'nøyaktighet': 0.788, 'makro-F1 (ekte)': 0.771, 'makro-F1 (trening)': 0.942}
{'modell': 'fritekst min_frequency=10', 'makro-F1': 0.724

,modell,makro-F1,nøyaktighet,makro-F1 (ekte),makro-F1 (trening)
0,fritekst min_frequency=2,0.730,0.786,0.764,0.945
1,fritekst min_frequency=5,0.729,0.788,0.771,0.942
2,uten nye variabler,0.728,0.789,0.773,0.942
3,uten manglende indikatorer,0.727,0.788,0.769,0.948
4,fritekst min_frequency=20,0.726,0.783,0.766,0.935
5,fritekst min_frequency=3,0.724,0.783,0.763,0.944
6,fritekst min_frequency=10,0.724,0.786,0.769,0.939
7,fritekst min_frequency=50,0.720,0.779,0.756,0.929
8,uten ord fra setningene,0.696,0.762,0.725,0.888
9,uten fritekst-svar (one-hot),0.682,0.752,0.713,0.912


Beste variant: fritekst min_frequency=2


**Funn:** Det er de samme variabelgruppene som betyr mest som i oppgave A, uten fritekstsvarene eller uten
ordene fra setningene synker makro-F1. Lavere `min_frequency` hjelper litt, og
`min_frequency = 2` gir best makro-F1 på valideringsdata (0,730).

### 7.7 Endelig modell, én evaluering på testdata

In [101]:
final_model_b = chosen_make_b(final_cfg_b)
# Tren modellen på slått sammen data
X_trval_b = pd.concat([X_train_b, X_val_b])
y_trval_b = pd.concat([y_train_b, y_val_b])
final_model_b.fit(X_trval_b, y_trval_b)
test_pred_b = final_model_b.predict(X_test_b)

real_test_b = (test_df_b["type"] == "real").to_numpy()
test_scores_b = pd.Series({
    "makro-F1": f1_score(y_test_b, test_pred_b, average="macro"),
    "nøyaktighet": accuracy_score(y_test_b, test_pred_b),
    "makro-F1 (ekte)": f1_score(y_test_b[real_test_b], test_pred_b[real_test_b], average="macro"),
}, name=f"{chosen_model_b} | {best_variant_b}")
display(test_scores_b.round(3).to_frame())

,"SVM, C = 3, class_weight=balanced | fritekst min_frequency=2"
makro-F1,0.707
nøyaktighet,0.781
makro-F1 (ekte),0.755


In [102]:
report_b = pd.DataFrame(classification_report(y_test_b, test_pred_b, output_dict=True)).T
display(report_b.loc[CLASSES].round(2))

fig = px.bar(report_b.loc[CLASSES].reset_index(), x="index", y="f1-score",
             title="F1 per klasse på testdata", labels={"index": "yrkeskategori", "f1-score": "F1"})
fig.show()

cm = confusion_matrix(y_test_b, test_pred_b, labels=CLASSES, normalize="true")
fig = px.imshow(cm, x=CLASSES, y=CLASSES, text_auto=".2f", color_continuous_scale="Blues", zmin=0, zmax=1,
                labels={"x": "predikert", "y": "fasit", "color": "andel"},
                title="Forvekslingsmatrise på testdata (normalisert per rad)", height=750)
fig.show()

,precision,recall,f1-score,support
politics & rule,0.92,0.85,0.89,354.0
performing arts,0.83,0.81,0.82,140.0
science & medicine,0.90,0.87,0.88,119.0
writing,0.76,0.71,0.73,87.0
philosophy & scholarship,0.60,0.68,0.64,65.0
other,0.50,0.64,0.56,64.0
visual arts & design,0.74,0.80,0.77,50.0
military,0.60,0.66,0.63,41.0
sport,0.65,0.51,0.57,39.0
famous by association,0.51,0.68,0.58,37.0


**Funn:** Modellen får makro-F1 0,707 på testdata, mot 0,730 på validering. Et lite fall er forventet, siden
modellen ble valgt fordi den hadde høyest skår av mange kandidater på valideringsdataene. Den generaliserer
altså godt, og er langt bedre enn grunnlinjen (0,052).

Diagonalen i forvekslingsmatrisen viser hvor stor andel av hver klasse som klassifiseres riktig (dekning).
`science & medicine` (87 %), `politics & rule` (85 %) og `performing arts` (81 %) treffes best, mens `sport`
er vanskeligst (51 %). Feilene er ikke tilfeldige, men følger klasser som ligner hverandre:

- 20 % av `military` predikeres som `politics & rule`, naturlig siden mange generaler og hærførere også var
  statsledere.
- 26 % av `sport` havner i samlekategorien `other`.
- `philosophy & scholarship` og `writing` blandes med hverandre og med `science & medicine`, som
  passer med at mange historiske tenkere både skrev, filosoferte og drev med vitenskap.



## 8. Oppsummering

In [103]:
summary = pd.DataFrame({
    "oppgave": ["A: born (regresjon)", "B: occupation_category (klassifikasjon)"],
    "endelig modell": [f"{best_name} | {best_variant}", f"{chosen_model_b} | {best_variant_b}"],
    "mål": ["MAE (år)", "makro-F1"],
    "test": [round(mean_absolute_error(y_test, test_pred), 1),
             round(f1_score(y_test_b, test_pred_b, average="macro"), 3)],
    "grunnlinje (validering)": [round(res.loc[res["modell"] == "Baseline: median", "MAE"].iloc[0], 1),
                                round(res_b.loc[res_b["modell"] == "Baseline: vanligste klasse", "makro-F1"].iloc[0], 3)],
}).set_index("oppgave")
display(summary)

,endelig modell,mål,test,grunnlinje (validering)
oppgave,,,,
A: born (regresjon),"Lasso, log-alder, alpha=0.0003: | fritekst min...",MAE (år),351.300,453.400
B: occupation_category (klassifikasjon),"SVM, C = 3, class_weight=balanced | fritekst m...",makro-F1,0.707,0.052


Begge de endelige modellene slår grunnlinjene tydelig. Fødselsår predikeres godt for den typiske personen, men
bommer mye for fiktive figurer og personer fra oldtiden, der også fasiten er minst pålitelig. Yrkeskategori
predikeres godt for store, tydelige klasser og dårligst for de små samlekategoriene.